In [ ]:
from posebusters.posebusters import PoseBusters
from rdkit import Chem
import os
import pandas as pd
import re 
from typing import List

import pandas as pd
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from matplotlib.colors import LinearSegmentedColormap

# Load data

In [ ]:
from Approach import DiffDockApproach, DiffDockPocketApproach, GninaApproach, SurfDockApproach, ICMApproach, ChaiApproach
exp_name = "plinder_set_0"
approaches = [
    DiffDockApproach(),
    DiffDockPocketApproach(),
    GninaApproach(),
    SurfDockApproach(),
    ICMApproach(),
    ChaiApproach(),
]
df_all = []
for approach in approaches:
    method_name = approach.get_name()
    print(method_name)
    df_method  = pd.read_csv(f"{method_name}_{exp_name}_results.csv")
    print(df_method.shape)
    df_all.append(df_method)

df_combined = pd.concat(df_all, ignore_index=True)

In [ ]:
df_combined['rmsd_≤_1å'] = df_combined['rmsd'] <=1
df_combined['rmsd_≤_5å'] = df_combined['rmsd'] <=5

In [ ]:
print("number of unique protein-ligand complexes from surfdock", len(df_combined[df_combined.method == 'surfdock']['protein'].unique()))

## load the annotation data

In [ ]:
test_annot_df = pd.read_csv('plinder_test.csv')
print(test_annot_df.shape)

In [ ]:
icm_df = df_combined[df_combined.method == 'icm']
diff_proteins = set(icm_df['protein'].unique()).difference(set(test_annot_df['system_id'].unique()))
print(diff_proteins)
print(icm_df['protein'].unique())

diff_cols = set(icm_df.columns).intersection(set(test_annot_df.columns))
print("Number of common columns:", len(diff_cols))
print("Common columns:", diff_cols)


In [ ]:
# Group by protein and find the row with minimum RMSD for each protein
best_rmsd_df = icm_df.groupby('protein').apply(
    lambda x: x.loc[x['rmsd'].idxmin()] if not x['rmsd'].isna().all() else x.iloc[0]
).reset_index(drop=True)

# Count how many proteins have RMSD ≤ 1Å and ≤ 5Å
count_rmsd_under_1 = best_rmsd_df['rmsd_≤_1å'].sum()
count_rmsd_under_5 = best_rmsd_df['rmsd_≤_5å'].sum()
total_proteins = len(best_rmsd_df)
percentage_under_1 = (count_rmsd_under_1 / total_proteins) * 100
percentage_under_5 = (count_rmsd_under_5 / total_proteins) * 100

# Print summary statistics
print(f"Total number of unique proteins: {total_proteins}")
print(f"Number of proteins with best RMSD ≤ 1Å: {count_rmsd_under_1} ({percentage_under_1:.2f}%)")
print(f"Number of proteins with best RMSD ≤ 5Å: {count_rmsd_under_5} ({percentage_under_5:.2f}%)")

# Show distribution of best RMSDs
plt.figure(figsize=(10, 6))
sns.histplot(best_rmsd_df['rmsd'].dropna(), bins=20, kde=True)
plt.axvline(1.0, color='red', linestyle='--', label='RMSD = 1Å threshold')
plt.axvline(5.0, color='orange', linestyle='--', label='RMSD = 5Å threshold')
plt.xlabel('Best RMSD per protein')
plt.ylabel('Count')
plt.title('Distribution of Best RMSD Values per Protein')
plt.legend()
plt.show()

# Return the dataframe for further analysis
best_rmsd_df.head()


In [ ]:
# Merge the dataframes
test_annot_df_unique = test_annot_df.drop_duplicates(subset=['system_id'])

merged_df = pd.merge(
    best_rmsd_df,
    test_annot_df_unique,
    left_on='protein',
    right_on='system_id',
    how='left'  # Use 'left' to keep all rows from icm_df
)

# See the result
print(f"best_rmsd_df shape: {best_rmsd_df.shape}")
print(f"test_annot_df_unique shape: {test_annot_df_unique.shape}")
print(f"merged_df shape: {merged_df.shape}")

# Check first few rows
merged_df.head()

In [ ]:
# Count duplicates across all columns
exact_duplicates = test_annot_df.duplicated().sum()
print(f"Number of exact duplicates in test_annot_df: {exact_duplicates}")

# To identify which rows are duplicated
duplicate_rows = test_annot_df[test_annot_df.duplicated(keep='first')]
print(f"Number of duplicate rows: {len(duplicate_rows)}")

# Let's check if there are duplicates when considering only system_id
system_id_duplicates = test_annot_df.duplicated(subset=['system_id'], keep=False)
print(f"Number of rows with duplicate system_id: {system_id_duplicates.sum()}")

# Show the duplicate system_ids
duplicate_systems = test_annot_df[system_id_duplicates].sort_values('system_id')
print("\nCount of each duplicate system_id:")
print(test_annot_df[system_id_duplicates]['system_id'].value_counts().head())

# Show a few examples of duplicate system_ids
if system_id_duplicates.sum() > 0:
    print("\nExample of duplicate system_ids:")
    sample_id = duplicate_systems['system_id'].iloc[0]
    sample_rows = test_annot_df[test_annot_df['system_id'] == sample_id]
    
    # Check if these duplicates are identical in all other columns
    is_identical = sample_rows.iloc[0].equals(sample_rows.iloc[1])
    print(f"Are the duplicate rows for {sample_id} identical in all columns? {is_identical}")
    
    # If not identical, show which columns differ
    if not is_identical:
        differing_columns = [col for col in sample_rows.columns if not sample_rows[col].nunique() == 1]
        print(f"Columns with different values: {differing_columns[:10]} {'...' if len(differing_columns) > 10 else ''}")

In [ ]:
print(test_annot_df['system_id'].unique())
test_annot_df['system_id'].nunique()
test_annot_df['system_id'].value_counts().head(10)

## merge with split df containing ions and other info

In [ ]:
merged_df.shape

In [ ]:
inputs_csv = pd.read_csv('./plidner_test_input.csv')
intersection_columns = set(inputs_csv.columns).intersection(set(test_annot_df.columns))
print(inputs_csv.shape, intersection_columns)
# droped_df = inputs_csv.drop(columns=list(intersection_columns.remove('system_id')), inplace=False)
# print(inputs_csv.drop(columns=intersection_columns.remove('system_id'), inplace=False).shape)
intersection_columns.remove('system_id')
merged_df = pd.merge(
    merged_df,
    inputs_csv.drop(columns=list(intersection_columns)), 
    left_on='protein',
    right_on='system_id',
    how='left'  # Use 'left' to keep all rows from icm_df
)
print(merged_df.shape)

# Analysis

## Resolution 

In [ ]:
from scipy import stats

# Analyzing the impact of resolution on RMSD

# First, let's ensure we're working with clean data
analysis_df = merged_df.copy()
analysis_df = analysis_df.dropna(subset=['rmsd', 'entry_resolution'])

# Create a scatter plot with regression line
plt.figure(figsize=(12, 8))
sns.scatterplot(data=analysis_df, x='entry_resolution', y='rmsd', alpha=0.6)
sns.regplot(data=analysis_df, x='entry_resolution', y='rmsd', scatter=False, color='red')

# Add horizontal lines for RMSD thresholds
plt.axhline(y=1, color='green', linestyle='--', label='RMSD = 1Å')
plt.axhline(y=5, color='orange', linestyle='--', label='RMSD = 5Å')

plt.title('Impact of Crystal Structure Resolution on Docking RMSD', fontsize=16)
plt.xlabel('Resolution (Å)', fontsize=14)
plt.ylabel('RMSD (Å)', fontsize=14)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Let's categorize resolution into bins for boxplot analysis
analysis_df['resolution_category'] = pd.cut(
    analysis_df['entry_resolution'], 
    bins=[0, 1.5, 2.0, 2.5, 3.0, float('inf')],
    labels=['≤1.5Å', '1.5-2.0Å', '2.0-2.5Å', '2.5-3.0Å', '>3.0Å']
)

# Create boxplot by resolution category
plt.figure(figsize=(14, 8))
sns.boxplot(data=analysis_df, x='resolution_category', y='rmsd')
plt.title('Distribution of RMSD by Resolution Category', fontsize=16)
plt.xlabel('Resolution Category', fontsize=14)
plt.ylabel('RMSD (Å)', fontsize=14)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Calculate success rate (RMSD ≤ 1Å and ≤ 5Å) by resolution category
success_by_resolution = analysis_df.groupby('resolution_category').agg(
    total_count=('rmsd', 'count'),
    success_1A=('rmsd_≤_1å', 'sum'),
    success_5A=('rmsd_≤_5å', 'sum')
).reset_index()

success_by_resolution['success_rate_1A'] = (success_by_resolution['success_1A'] / success_by_resolution['total_count']) * 100
success_by_resolution['success_rate_5A'] = (success_by_resolution['success_5A'] / success_by_resolution['total_count']) * 100

# Plot success rates by resolution category
plt.figure(figsize=(14, 8))
bar_width = 0.35
index = np.arange(len(success_by_resolution['resolution_category']))

plt.bar(index - bar_width/2, success_by_resolution['success_rate_1A'], bar_width, label='RMSD ≤ 1Å', color='green', alpha=0.7)
plt.bar(index + bar_width/2, success_by_resolution['success_rate_5A'], bar_width, label='RMSD ≤ 5Å', color='orange', alpha=0.7)

plt.xlabel('Resolution Category', fontsize=14)
plt.ylabel('Success Rate (%)', fontsize=14)
plt.title('Docking Success Rate by Resolution Category', fontsize=16)
plt.xticks(index, success_by_resolution['resolution_category'])
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Statistical analysis

# Correlation analysis
correlation, p_value = stats.pearsonr(analysis_df['entry_resolution'], analysis_df['rmsd'])
print(f"Pearson correlation between resolution and RMSD: {correlation:.4f}")
print(f"P-value: {p_value:.6f}")

# Calculate average RMSD by resolution category
avg_rmsd_by_resolution = analysis_df.groupby('resolution_category')['rmsd'].agg(['mean', 'median', 'std', 'count']).reset_index()
print("\nAverage RMSD by resolution category:")
print(avg_rmsd_by_resolution)

# One-way ANOVA to test if resolution categories have significantly different RMSDs
groups = [analysis_df[analysis_df['resolution_category'] == cat]['rmsd'] for cat in analysis_df['resolution_category'].unique()]
f_stat, anova_p = stats.f_oneway(*[group for group in groups if len(group) > 0])
print(f"\nOne-way ANOVA: F-statistic = {f_stat:.4f}, p-value = {anova_p:.6f}")

if anova_p < 0.05:
    print("The resolution categories have significantly different RMSD distributions.")
else:
    print("The resolution categories do not have significantly different RMSD distributions.")

## analysis regarding if ion present in the pocket

In [ ]:
import ast

# Check the current type
print(f"Type before conversion: {type(merged_df['ion_paths'].iloc[0])}")
print(f"Sample value: {merged_df['ion_paths'].iloc[0]}")

# Convert strings to lists using ast.literal_eval
merged_df['ion_paths'] = merged_df['ion_paths'].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)

# Check the result
print(f"Type after conversion: {type(merged_df['ion_paths'].iloc[0])}")
print(f"Sample value: {merged_df['ion_paths'].iloc[0]}")

# Now you can create the has_ion column correctly
merged_df['has_ion'] = merged_df['ion_paths'].apply(lambda x: len(x) > 0 if isinstance(x, list) else False)

# Check the distribution of has_ion
print(merged_df['has_ion'].value_counts())

In [ ]:
from scipy import stats

# Extract information about ion presence
merged_df['has_ion'] = merged_df['ion_paths'].apply(lambda x: len(x) > 0 if isinstance(x, list) else False)

# Create a DataFrame for statistical analysis
stats_df = merged_df.groupby('has_ion').agg(
    total_count=('rmsd', 'count'),
    rmsd_mean=('rmsd', 'mean'),
    rmsd_median=('rmsd', 'median'),
    rmsd_std=('rmsd', 'std'),
    success_1A=('rmsd_≤_1å', 'sum'),
    success_2A=('rmsd_≤_2å', 'sum')
).reset_index()

# Calculate success rates
stats_df['success_rate_1A'] = (stats_df['success_1A'] / stats_df['total_count']) * 100
stats_df['success_rate_2A'] = (stats_df['success_2A'] / stats_df['total_count']) * 100

# Display statistics
print("Statistics by ion presence:")
print(stats_df)

# Create a new column for RMSD categories
merged_df['rmsd_category'] = pd.cut(
    merged_df['rmsd'], 
    bins=[0, 1, 2, 5, float('inf')],
    labels=['≤1Å', '1-2Å', '2-5Å', '>5Å']
)

# Perform statistical test to see if ion presence affects RMSD

# T-test for RMSD values
ion_rmsd = merged_df[merged_df['has_ion']]['rmsd'].dropna()
no_ion_rmsd = merged_df[~merged_df['has_ion']]['rmsd'].dropna()
t_stat, p_value = stats.ttest_ind(ion_rmsd, no_ion_rmsd, equal_var=False)
print(f"\nT-test for RMSD by ion presence: t={t_stat:.4f}, p-value={p_value:.4f}")
if p_value < 0.05:
    print("The difference in RMSD between systems with and without ions is statistically significant.")
else:
    print("No statistically significant difference in RMSD between systems with and without ions.")

# Chi-square test for success rates
contingency_table = pd.crosstab(merged_df['has_ion'], merged_df['rmsd_category'])
chi2, p, dof, expected = stats.chi2_contingency(contingency_table)
print(f"\nChi-square test for RMSD categories by ion presence: chi2={chi2:.4f}, p-value={p:.4f}")
if p < 0.05:
    print("The distribution of RMSD categories differs significantly based on ion presence.")
else:
    print("No significant difference in the distribution of RMSD categories based on ion presence.")

# Visualize the results
plt.figure(figsize=(15, 10))

# Plot 1: Box plot of RMSD by ion presence
plt.subplot(2, 2, 1)
sns.boxplot(x='has_ion', y='rmsd', data=merged_df)
plt.title('RMSD Distribution by Ion Presence')
plt.xlabel('Ion Present')
plt.ylabel('RMSD (Å)')
plt.ylim(0, 15)  # Limit y-axis for better visualization

# Plot 2: Bar plot of success rates
plt.subplot(2, 2, 2)
bar_width = 0.35
index = np.arange(len(stats_df))
plt.bar(index - bar_width/2, stats_df['success_rate_1A'], bar_width, label='RMSD ≤ 1Å', color='green')
plt.bar(index + bar_width/2, stats_df['success_rate_2A'], bar_width, label='RMSD ≤ 2Å', color='orange')
plt.xlabel('Ion Present')
plt.ylabel('Success Rate (%)')
plt.title('Docking Success Rate by Ion Presence')
plt.xticks(index, ['No', 'Yes'])
plt.legend()

# Plot 3: Distribution of RMSD categories
plt.subplot(2, 2, 3)
rmsd_cat_counts = merged_df.groupby(['has_ion', 'rmsd_category']).size().unstack()
rmsd_cat_pct = rmsd_cat_counts.div(rmsd_cat_counts.sum(axis=1), axis=0) * 100
rmsd_cat_pct.plot(kind='bar', stacked=True, ax=plt.gca())
plt.title('Distribution of RMSD Categories by Ion Presence')
plt.xlabel('Ion Present')
plt.ylabel('Percentage (%)')
plt.legend(title='RMSD Category')

# Plot 4: Histogram of RMSDs
plt.subplot(2, 2, 4)
sns.histplot(data=merged_df, x='rmsd', hue='has_ion', element='step', stat='density', common_norm=False, bins=30)
plt.title('RMSD Distribution by Ion Presence')
plt.xlabel('RMSD (Å)')
plt.ylabel('Density')
plt.xlim(0, 15)  # Limit x-axis for better visualization

plt.tight_layout()
plt.show()

# Calculate effect size (Cohen's d)
mean_ion = ion_rmsd.mean()
mean_no_ion = no_ion_rmsd.mean()
std_pooled = np.sqrt((ion_rmsd.var() * (len(ion_rmsd) - 1) + no_ion_rmsd.var() * (len(no_ion_rmsd) - 1)) / 
                     (len(ion_rmsd) + len(no_ion_rmsd) - 2))
cohen_d = abs(mean_ion - mean_no_ion) / std_pooled

print(f"\nEffect size (Cohen's d): {cohen_d:.4f}")
if cohen_d < 0.2:
    print("The effect size is small.")
elif cohen_d < 0.5:
    print("The effect size is medium.")
else:
    print("The effect size is large.")

## analysis regarding protein

### entry_validation_molprobity

In [ ]:
def analyze_rmsd_property_correlation(df, property_name, property_type='continuous', title=None, bins=None, plot=True):
    """
    Analyze the correlation between RMSD and a specified property (continuous, discrete, or binary).
    
    Parameters:
    -----------
    df : pandas.DataFrame
        DataFrame containing RMSD and the property of interest
    property_name : str
        Name of the column containing the property to analyze
    property_type : str, optional
        Type of property: 'continuous', 'discrete', or 'binary'
    title : str, optional
        Custom title for plots. Default uses property_name
    bins : list, optional
        Custom bins for categorizing continuous properties. If None, auto-creates 5 bins
    plot : bool, optional
        Whether to generate visualization plots
        
    Returns:
    --------
    dict
        Results of the analysis including correlations, statistical tests, and success rates
    """
    from scipy import stats
    import numpy as np
    import matplotlib.pyplot as plt
    import seaborn as sns
    import pandas as pd
    
    # Create a clean working copy
    analysis_df = df.copy()
    analysis_df = analysis_df.dropna(subset=['rmsd', property_name])
    
    if title is None:
        title = f'Impact of {property_name} on Docking RMSD'
    
    # Ensure RMSD threshold columns exist
    if 'rmsd_≤_1å' not in analysis_df.columns:
        analysis_df['rmsd_≤_1å'] = analysis_df['rmsd'] <= 1
    if 'rmsd_≤_2å' not in analysis_df.columns:
        analysis_df['rmsd_≤_2å'] = analysis_df['rmsd'] <= 2
    
    # Initialize result containers
    results = {}
    property_category = None
    
    # Handle different property types
    if property_type == 'continuous':
        # Calculate correlations for continuous data
        correlation, p_value = stats.pearsonr(analysis_df[property_name], analysis_df['rmsd'])
        spearman_corr, spearman_p = stats.spearmanr(analysis_df[property_name], analysis_df['rmsd'])
        
        results['pearson_correlation'] = correlation
        results['pearson_p_value'] = p_value
        results['spearman_correlation'] = spearman_corr
        results['spearman_p_value'] = spearman_p
        
        # Create property categories for continuous data
        if bins is None:
            # Auto-create 5 bins
            min_val = analysis_df[property_name].min()
            max_val = analysis_df[property_name].max()
            range_val = max_val - min_val
            step = range_val / 5
            bins = [min_val, min_val+step, min_val+2*step, min_val+3*step, min_val+4*step, float('inf')]
            bin_labels = [f'≤{min_val+step:.1f}', f'{min_val+step:.1f}-{min_val+2*step:.1f}', 
                         f'{min_val+2*step:.1f}-{min_val+3*step:.1f}', f'{min_val+3*step:.1f}-{min_val+4*step:.1f}', f'>{min_val+4*step:.1f}']
        else:
            bin_labels = [f'≤{bins[1]}', f'{bins[1]}-{bins[2]}', f'{bins[2]}-{bins[3]}', f'{bins[3]}-{bins[4]}', f'>{bins[4]}']
        
        property_category = f'{property_name}_category'
        analysis_df[property_category] = pd.cut(
            analysis_df[property_name], 
            bins=bins,
            labels=bin_labels
        )
        
        # Print key statistics for continuous data
        print(f"Pearson correlation between {property_name} and RMSD: {correlation:.4f}")
        print(f"P-value: {p_value:.6f}")
        print(f"Spearman correlation: {spearman_corr:.4f}, p-value: {spearman_p:.6f}")
        
        # Visualizations for continuous data
        if plot:
            # Create a grid of plots: 1 row, 3 columns (scatter, boxplot, bar chart)
            fig, axes = plt.subplots(1, 3, figsize=(20, 6))
            fig.suptitle(title, fontsize=18)
            
            # 1. Scatter plot with regression line
            sns.scatterplot(data=analysis_df, x=property_name, y='rmsd', alpha=0.6, ax=axes[0])
            sns.regplot(data=analysis_df, x=property_name, y='rmsd', scatter=False, color='red', ax=axes[0])
            axes[0].axhline(y=1, color='green', linestyle='--', label='RMSD = 1Å')
            axes[0].axhline(y=2, color='orange', linestyle='--', label='RMSD = 2Å')
            axes[0].set_xlabel(f'{property_name}', fontsize=12)
            axes[0].set_ylabel('RMSD (Å)', fontsize=12)
            axes[0].set_title('Correlation Plot', fontsize=14)
            axes[0].legend()
            axes[0].grid(True, alpha=0.3)
            
            # 2. Boxplot by category
            sns.boxplot(data=analysis_df, x=property_category, y='rmsd', ax=axes[1])
            axes[1].set_xlabel(f'{property_name} Category', fontsize=12)
            axes[1].set_ylabel('RMSD (Å)', fontsize=12)
            axes[1].set_title('RMSD Distribution by Category', fontsize=14)
            axes[1].tick_params(axis='x', rotation=45)
            axes[1].grid(True, alpha=0.3)
            
            # 3. Success rate bar plot
            success_by_category = analysis_df.groupby(property_category).agg(
                total_count=('rmsd', 'count'),
                success_1A=('rmsd_≤_1å', 'sum'),
                success_2A=('rmsd_≤_2å', 'sum')
            ).reset_index()
            
            success_by_category['success_rate_1A'] = (success_by_category['success_1A'] / success_by_category['total_count']) * 100
            success_by_category['success_rate_2A'] = (success_by_category['success_2A'] / success_by_category['total_count']) * 100
            
            bar_width = 0.35
            index = np.arange(len(success_by_category[property_category]))
            
            # Create bars with counts on top
            bars1 = axes[2].bar(index - bar_width/2, success_by_category['success_rate_1A'], 
                    bar_width, label='RMSD ≤ 1Å', color='green', alpha=0.7)
            bars2 = axes[2].bar(index + bar_width/2, success_by_category['success_rate_2A'], 
                    bar_width, label='RMSD ≤ 2Å', color='orange', alpha=0.7)
            
            # Add count labels on top of bars
            for i, bar in enumerate(bars1):
                count = int(success_by_category['total_count'].iloc[i])
                axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2, 
                         f'n={count}', ha='center', va='bottom', fontsize=9)
            
            axes[2].set_xlabel(f'{property_name} Category', fontsize=12)
            axes[2].set_ylabel('Success Rate (%)', fontsize=12)
            axes[2].set_title('Docking Success Rate by Category', fontsize=14)
            axes[2].set_xticks(index)
            axes[2].set_xticklabels(success_by_category[property_category])
            axes[2].tick_params(axis='x', rotation=45)
            axes[2].legend()
            axes[2].grid(True, alpha=0.3)
            
            plt.tight_layout()
            plt.subplots_adjust(top=0.9)  # Adjust for the suptitle
            plt.show()
            
    elif property_type == 'discrete':
        # For discrete numerical data (e.g., counts, integers)
        property_category = property_name  # Use the property itself as category
        
        # Calculate non-parametric correlation for discrete data
        spearman_corr, spearman_p = stats.spearmanr(analysis_df[property_name], analysis_df['rmsd'])
        results['spearman_correlation'] = spearman_corr
        results['spearman_p_value'] = spearman_p
        
        # Print key statistics for discrete data
        print(f"Spearman correlation between {property_name} and RMSD: {spearman_corr:.4f}")
        print(f"P-value: {spearman_p:.6f}")
        
        # Visualizations for discrete data
        if plot:
            # Create a grid of plots: 1 row, 3 columns (box plot, violin plot, success rate)
            fig, axes = plt.subplots(1, 3, figsize=(20, 6))
            fig.suptitle(title, fontsize=18)
            
            # 1. Box plot by discrete values
            sns.boxplot(data=analysis_df, x=property_name, y='rmsd', ax=axes[0])
            axes[0].set_title('RMSD Distribution (Box Plot)', fontsize=14)
            axes[0].set_xlabel(f'{property_name}', fontsize=12)
            axes[0].set_ylabel('RMSD (Å)', fontsize=12)
            axes[0].grid(True, alpha=0.3)
            
            # 2. Violin plot
            sns.violinplot(data=analysis_df, x=property_name, y='rmsd', ax=axes[1])
            axes[1].set_title('RMSD Distribution (Violin Plot)', fontsize=14)
            axes[1].set_xlabel(f'{property_name}', fontsize=12)
            axes[1].set_ylabel('RMSD (Å)', fontsize=12)
            axes[1].grid(True, alpha=0.3)
            
            # 3. Success rate bar plot
            success_by_category = analysis_df.groupby(property_category).agg(
                total_count=('rmsd', 'count'),
                success_1A=('rmsd_≤_1å', 'sum'),
                success_2A=('rmsd_≤_2å', 'sum')
            ).reset_index()
            
            success_by_category['success_rate_1A'] = (success_by_category['success_1A'] / success_by_category['total_count']) * 100
            success_by_category['success_rate_2A'] = (success_by_category['success_2A'] / success_by_category['total_count']) * 100
            
            bar_width = 0.35
            index = np.arange(len(success_by_category[property_category]))
            
            # Create bars with counts on top
            bars1 = axes[2].bar(index - bar_width/2, success_by_category['success_rate_1A'], 
                    bar_width, label='RMSD ≤ 1Å', color='green', alpha=0.7)
            bars2 = axes[2].bar(index + bar_width/2, success_by_category['success_rate_2A'], 
                    bar_width, label='RMSD ≤ 2Å', color='orange', alpha=0.7)
            
            # Add count labels on top of bars
            for i, bar in enumerate(bars1):
                count = int(success_by_category['total_count'].iloc[i])
                axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2, 
                         f'n={count}', ha='center', va='bottom', fontsize=9)
            
            axes[2].set_xlabel(f'{property_name}', fontsize=12)
            axes[2].set_ylabel('Success Rate (%)', fontsize=12)
            axes[2].set_title('Docking Success Rate', fontsize=14)
            axes[2].set_xticks(index)
            axes[2].set_xticklabels(success_by_category[property_category])
            axes[2].legend()
            axes[2].grid(True, alpha=0.3)
            
            plt.tight_layout()
            plt.subplots_adjust(top=0.9)  # Adjust for the suptitle
            plt.show()
            
    elif property_type == 'binary':
        # For binary data (True/False, 0/1)
        property_category = property_name  # Use the property itself as category
        
        # Ensure the property is treated as boolean/binary
        if analysis_df[property_name].dtype != bool:
            # Assume 0/1 or string 'True'/'False' needs conversion
            if analysis_df[property_name].dtype == 'object':
                analysis_df[property_name] = analysis_df[property_name].astype(str).str.lower() == 'true'
            else:
                analysis_df[property_name] = analysis_df[property_name].astype(bool)
        
        # Point-biserial correlation for binary data
        # Convert boolean to 0/1 for correlation
        binary_values = analysis_df[property_name].astype(int)
        point_biserial, pb_pvalue = stats.pointbiserialr(binary_values, analysis_df['rmsd'])
        results['point_biserial_correlation'] = point_biserial
        results['point_biserial_p_value'] = pb_pvalue
        
        # Mann-Whitney U test for RMSD difference between groups
        true_rmsd = analysis_df[analysis_df[property_name]]['rmsd']
        false_rmsd = analysis_df[~analysis_df[property_name]]['rmsd']
        if len(true_rmsd) > 0 and len(false_rmsd) > 0:  # Ensure both groups have data
            mw_stat, mw_pvalue = stats.mannwhitneyu(true_rmsd, false_rmsd, alternative='two-sided')
            results['mannwhitney_stat'] = mw_stat
            results['mannwhitney_p_value'] = mw_pvalue
        
        # Print key statistics for binary data
        print(f"Point-biserial correlation between {property_name} and RMSD: {point_biserial:.4f}")
        print(f"P-value: {pb_pvalue:.6f}")
        
        if 'mannwhitney_p_value' in results:
            print(f"Mann-Whitney U test: p-value = {mw_pvalue:.6f}")
            if mw_pvalue < 0.05:
                print(f"The RMSD distributions differ significantly based on {property_name}.")
            else:
                print(f"No significant difference in RMSD distributions based on {property_name}.")
        
        # Visualizations for binary data
        if plot:
            # Create a grid of plots: 1 row, 3 columns
            fig, axes = plt.subplots(1, 3, figsize=(20, 6))
            fig.suptitle(title, fontsize=18)
            
            # 1. Box plot by binary values
            sns.boxplot(x=property_name, y='rmsd', data=analysis_df, ax=axes[0])
            axes[0].set_title('RMSD Distribution', fontsize=14)
            axes[0].set_xlabel(f'{property_name}', fontsize=12)
            axes[0].set_ylabel('RMSD (Å)', fontsize=12)
            axes[0].grid(True, alpha=0.3)
            
            # 2. Histogram for RMSD distribution by group
            sns.histplot(data=analysis_df, x='rmsd', hue=property_name, element='step', 
                         stat='density', common_norm=False, bins=30, ax=axes[1])
            axes[1].set_title('RMSD Density by Group', fontsize=14)
            axes[1].set_xlabel('RMSD (Å)', fontsize=12)
            axes[1].set_ylabel('Density', fontsize=12)
            axes[1].axvline(x=1, color='green', linestyle='--', label='RMSD = 1Å')
            axes[1].axvline(x=2, color='orange', linestyle='--', label='RMSD = 2Å')
            axes[1].legend()
            axes[1].grid(True, alpha=0.3)
            
            # 3. Bar chart of success rates by group
            success_by_binary = analysis_df.groupby(property_name).agg(
                total_count=('rmsd', 'count'),
                success_1A=('rmsd_≤_1å', 'sum'),
                success_2A=('rmsd_≤_2å', 'sum')
            ).reset_index()
            
            success_by_binary['success_rate_1A'] = (success_by_binary['success_1A'] / success_by_binary['total_count']) * 100
            success_by_binary['success_rate_2A'] = (success_by_binary['success_2A'] / success_by_binary['total_count']) * 100
            
            bar_width = 0.35
            index = np.arange(len(success_by_binary))
            
            # Create bars with counts on top
            bars1 = axes[2].bar(index - bar_width/2, success_by_binary['success_rate_1A'], 
                    bar_width, label='RMSD ≤ 1Å', color='green', alpha=0.7)
            bars2 = axes[2].bar(index + bar_width/2, success_by_binary['success_rate_2A'], 
                    bar_width, label='RMSD ≤ 2Å', color='orange', alpha=0.7)
            
            # Add count labels on top of bars
            for i, bar in enumerate(bars1):
                count = int(success_by_binary['total_count'].iloc[i])
                axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2, 
                         f'n={count}', ha='center', va='bottom', fontsize=9)
            
            axes[2].set_xlabel(f'{property_name}', fontsize=12)
            axes[2].set_ylabel('Success Rate (%)', fontsize=12)
            axes[2].set_title('Docking Success Rate', fontsize=14)
            axes[2].set_xticks(index)
            axes[2].set_xticklabels([str(x) for x in success_by_binary[property_name]])
            axes[2].legend()
            axes[2].grid(True, alpha=0.3)
            
            plt.tight_layout()
            plt.subplots_adjust(top=0.9)  # Adjust for the suptitle
            plt.show()
    
    # For all property types, calculate success rates by category (not required to plot again)
    success_by_category = analysis_df.groupby(property_category).agg(
        total_count=('rmsd', 'count'),
        success_1A=('rmsd_≤_1å', 'sum'),
        success_2A=('rmsd_≤_2å', 'sum')
    ).reset_index()
    
    success_by_category['success_rate_1A'] = (success_by_category['success_1A'] / success_by_category['total_count']) * 100
    success_by_category['success_rate_2A'] = (success_by_category['success_2A'] / success_by_category['total_count']) * 100
    
    # Calculate average RMSD by category
    avg_rmsd_by_category = analysis_df.groupby(property_category)['rmsd'].agg(
        ['mean', 'median', 'std', 'count']
    ).reset_index()
    
    # One-way ANOVA for all types (if at least 2 groups)
    groups = [analysis_df[analysis_df[property_category] == cat]['rmsd'] 
              for cat in analysis_df[property_category].unique()]
    valid_groups = [group for group in groups if len(group) > 0]
    if len(valid_groups) > 1:  # ANOVA requires at least 2 groups
        f_stat, anova_p = stats.f_oneway(*valid_groups)
        results['f_statistic'] = f_stat
        results['anova_p_value'] = anova_p
        
        if property_type == 'continuous' or property_type == 'discrete':
            print(f"\nOne-way ANOVA: F-statistic = {f_stat:.4f}, p-value = {anova_p:.6f}")
            if anova_p < 0.05:
                print(f"The {property_name} categories have significantly different RMSD distributions.")
            else:
                print(f"The {property_name} categories do not have significantly different RMSD distributions.")
    
    # Add success rates and averages to results
    results['success_by_category'] = success_by_category
    results['avg_rmsd_by_category'] = avg_rmsd_by_category
    results['analysis_df'] = analysis_df
    
    return results

In [ ]:
# Example usage for MolProbity score analysis
molprobity_results = analyze_rmsd_property_correlation(
    merged_df, 
    property_name='entry_validation_molprobity',
    title='Impact of MolProbity Score on Docking RMSD',
    property_type='continuous',  # or 'discrete', 'binary'
    bins=None  # Let the function auto-create bins, or specify your own like [0, 1, 2, 3, 4, float('inf')]
)

protein_results = analyze_rmsd_property_correlation(
    merged_df, 
    property_name='system_num_pocket_residues',
    title='Impact of Protein Residues on Docking RMSD',
    property_type='continuous',  # or 'discrete', 'binary'
)

protein_results = analyze_rmsd_property_correlation(
    merged_df, 
    property_name='system_num_interactions',
    title='Impact of Protein Interactions on Docking RMSD',
    property_type='continuous',  # or 'discrete', 'binary'
)

protein_results = analyze_rmsd_property_correlation(
    merged_df, 
    property_name='system_proper_ligand_max_molecular_weight',
    title='Impact of Ligand Molecular Weight on Docking RMSD',
    property_type='continuous',  # or 'discrete', 'binary'
)

entry_validation_results = analyze_rmsd_property_correlation(
    merged_df, 
    property_name='entry_validation_atom_count',
    title='Impact of Entry Validation Atom Count on Docking RMSD',
    property_type='continuous',  # or 'discrete', 'binary'
    bins=[0, 200, 400, 600, 800, float('inf')]
)

entry_validation_results = analyze_rmsd_property_correlation(
    merged_df, 
    property_name='system_num_protein_chains',
    title='Impact of Number of Protein Chains on Docking RMSD',
    property_type='discrete',  # or 'discrete', 'binary'
)

entry_validation_results = analyze_rmsd_property_correlation(
    merged_df, 
    property_name='system_pocket_validation_atom_count',
    title='Impact of Pocket Validation Atom Count on Docking RMSD',
    property_type='continuous',  # or 'discrete', 'binary'
    bins=[0, 200, 400, 600, 800, float('inf')]
)

## ligand properties

In [ ]:
# Example usage for MolProbity score analysis
cripped_clog_results = analyze_rmsd_property_correlation(
    merged_df, 
    property_name='ligand_crippen_clogp',
    title='Impact of Ligand Cropped ClogP on Docking RMSD',
    property_type='continuous',  # or 'discrete', 'binary'
    bins=None  # Let the function auto-create bins, or specify your own like [0, 1, 2, 3, 4, float('inf')]
)

In [ ]:
# Example usage for MolProbity score analysis
molprobity_results = analyze_rmsd_property_correlation(
    merged_df, 
    property_name='ligand_molecular_weight',
    property_type='continuous',  # or 'discrete', 'binary'
    title='Impact of Ligand Molecular Weight on Docking RMSD',
    bins=[0, 200, 400, 600, 800, float('inf')]
)

In [ ]:
# For continuous numerical properties (like resolution, MolProbity score)
discrete_results = analyze_rmsd_property_correlation(
    merged_df, 
    property_name='ligand_num_hba',
    property_type='continuous',
    title='Impact of Number of Hydrogen Bond Acceptors on Docking RMSD',  
)

# For discrete numerical properties (like atom count, number of hydrogen bond donors)
discrete_results = analyze_rmsd_property_correlation(
    merged_df, 
    property_name='ligand_num_hbd',
    property_type='continuous',
    title='Impact of Number of Hydrogen Bond Donors on Docking RMSD'
)

# For binary properties (like has_ion)
discrete_results = analyze_rmsd_property_correlation(
    merged_df, 
    property_name='ligand_num_rings',
    property_type='discrete',
    title='Impact of Number of Rings on Docking RMSD'
)

### ligand_is_covalent
- ligand_is_covalent
- ligand_is_ion
- ligand_is_cofactor
- ligand_is_artifact

In [ ]:
binary_results = analyze_rmsd_property_correlation(
    merged_df, 
    property_name='ligand_is_covalent',
    property_type='binary',
    title='Impact of Covalent Ligands on Docking RMSD'
)
merged_df['ligand_is_covalent'].value_counts()

In [ ]:
binary_results = analyze_rmsd_property_correlation(
    merged_df, 
    property_name='ligand_is_cofactor',
    property_type='binary',
    title='Impact of Cofactor Ligands on Docking RMSD'
)
merged_df['ligand_is_cofactor'].value_counts()

In [ ]:
binary_results = analyze_rmsd_property_correlation(
    merged_df, 
    property_name='ligand_is_artifact',
    property_type='binary',
    title='Impact of Ion Ligands on Docking RMSD'
)
merged_df['ligand_is_artifact'].value_counts()

## Interactions 

In [ ]:
properties = ["ligand_num_interacting_residues", "ligand_num_neighboring_residues", "ligand_num_interactions"]
bins_dict = {
    "ligand_num_interacting_residues": [0, 5, 10, 15, 20, 25],
    "ligand_num_neighboring_residues": [0, 10, 20, 30, 40, 50],
    "ligand_num_interactions": [0, 5, 10, 15, 20, 25],
}
for property in properties:
    discrete_results = analyze_rmsd_property_correlation(
        merged_df, 
        property_name=property,
        property_type='continuous',
        title=f'Impact of {property} on Docking RMSD',
        # bins=bins_dict[f"{property}"]
    )

# Analysis for other method

In [ ]:
protein_properties = ["entry_resolution", "entry_validation_molprobity", "system_num_pocket_residues", "system_num_interactions" ]
ligand_properties = ["ligand_crippen_clogp", "ligand_molecular_weight", "ligand_num_hba", "ligand_num_hbd", "ligand_num_rot_bonds"]
interaction_properties = ["ligand_num_interacting_residues", "ligand_num_neighboring_residues", "ligand_num_interactions"]
discrete_properties = ["system_num_protein_chains", "ligand_num_rings", ]
binary_properties = ["has_ion", "ligand_is_covalent", "ligand_is_cofactor"]
bins_dict = {
    "ligand_num_interacting_residues": [0, 5, 10, 15, 20, 25],
    "ligand_num_neighboring_residues": [0, 10, 20, 30, 40, 50],
    "ligand_num_interactions": [0, 5, 10, 15, 20, 25],
    "ligand_molecular_weight": [0, 100, 200, 300, 400, 500],
    "ligand_num_hba": [0, 1, 2, 3, 4, 5],
    "ligand_num_hbd": [0, 1, 2, 3, 4, 5],
    "ligand_num_rings": [0, 1, 2, 3, 4, 5],
}

In [ ]:
import ast
for method in df_combined['method'].unique():
    if method == 'icm' or method == 'diffdock':
        continue

    method_df = df_combined[df_combined['method'] == method]
    best_rmsd_df = method_df.groupby('protein').apply(
        lambda x: x.loc[x['rmsd'].idxmin()] if not x['rmsd'].isna().all() else x.iloc[0]
    ).reset_index(drop=True)
    method_success_rate = best_rmsd_df['rmsd_≤_2å'].mean() * 100
    print(f"Success rate for {method}: {method_success_rate:.2f}%")

    # Merge the dataframes
    test_annot_df_unique = test_annot_df.drop_duplicates(subset=['system_id'])

    merged_df = pd.merge(
        best_rmsd_df,
        test_annot_df_unique,
        left_on='protein',
        right_on='system_id',
        how='left'  # Use 'left' to keep all rows from icm_df
    )
    
    # See the result
    print(f"best_rmsd_df shape: {best_rmsd_df.shape}")
    print(f"test_annot_df_unique shape: {test_annot_df_unique.shape}")
    print(f"merged_df shape: {merged_df.shape}")

    intersection_columns = set(inputs_csv.columns).intersection(set(test_annot_df.columns))
    intersection_columns.remove('system_id')
    merged_df = pd.merge(
        merged_df,
        inputs_csv.drop(columns=list(intersection_columns)), 
        left_on='protein',
        right_on='system_id',
        how='left'  # Use 'left' to keep all rows from icm_df
    )
    
    # Convert strings to lists using ast.literal_eval
    merged_df['ion_paths'] = merged_df['ion_paths'].apply(
        lambda x: ast.literal_eval(x) if isinstance(x, str) else x
    )
    merged_df['has_ion'] = merged_df['ion_paths'].apply(lambda x: len(x) > 0 if isinstance(x, list) else False)
    print(f"analysis for {method} begins")
    
    for property in binary_properties:
        discrete_results = analyze_rmsd_property_correlation(
            merged_df, 
            property_name=property,
            property_type='binary',
            title=f'Impact of {property} on Docking RMSD',
            # bins=bins_dict[f"{property}"]
        )
    for properties in discrete_properties:
        discrete_results = analyze_rmsd_property_correlation(
            merged_df, 
            property_name=properties,
            property_type='discrete',
            title=f'Impact of {properties} on Docking RMSD',
            # bins=bins_dict[f"{properties}"]
        )
    for properties in protein_properties:
        discrete_results = analyze_rmsd_property_correlation(
            merged_df, 
            property_name=properties,
            property_type='continuous',
            title=f'Impact of {properties} on Docking RMSD',
            # bins=bins_dict[f"{properties}"]
        )
    for properties in ligand_properties:
        discrete_results = analyze_rmsd_property_correlation(
            merged_df, 
            property_name=properties,
            property_type='continuous',
            title=f'Impact of {properties} on Docking RMSD',
            # bins=bins_dict[f"{properties}"]
        )
    for properties in interaction_properties:
        discrete_results = analyze_rmsd_property_correlation(
            merged_df, 
            property_name=properties,
            property_type='continuous',
            title=f'Impact of {properties} on Docking RMSD',
            bins=bins_dict[f"{properties}"]
        )

# Comparative analysis

In [ ]:
def compare_method_performance(df_combined, method1, method2, rmsd_threshold=2.0):
    """
    Compare performance of two docking methods, identifying cases where one method 
    succeeds (RMSD <= threshold) and the other fails (RMSD > threshold).
    
    Parameters:
    -----------
    df_combined : pandas.DataFrame
        Combined dataframe with results from all methods
    method1 : str
        Name of the first method to compare
    method2 : str
        Name of the second method to compare
    rmsd_threshold : float, optional
        RMSD threshold for considering a docking successful, default is 2.0 Å
    
    Returns:
    --------
    dict
        Dictionary containing DataFrames of protein targets where one method succeeds
        and the other fails, as well as performance statistics
    """
    # Filter data for each method
    df1 = df_combined[df_combined['method'] == method1]
    df2 = df_combined[df_combined['method'] == method2]
    
    # Get best RMSD per protein for each method
    best_rmsd1 = df1.groupby('protein').apply(
        lambda x: x.loc[x['rmsd'].idxmin()] if not x['rmsd'].isna().all() else x.iloc[0]
    ).reset_index(drop=True)
    
    best_rmsd2 = df2.groupby('protein').apply(
        lambda x: x.loc[x['rmsd'].idxmin()] if not x['rmsd'].isna().all() else x.iloc[0]
    ).reset_index(drop=True)
    
    # Create column for success based on threshold
    best_rmsd1[f'rmsd_≤_{rmsd_threshold}å'] = best_rmsd1['rmsd'] <= rmsd_threshold
    best_rmsd2[f'rmsd_≤_{rmsd_threshold}å'] = best_rmsd2['rmsd'] <= rmsd_threshold
    
    # Combine data for comparison
    merged = pd.merge(
        best_rmsd1[['protein', 'rmsd', f'rmsd_≤_{rmsd_threshold}å']], 
        best_rmsd2[['protein', 'rmsd', f'rmsd_≤_{rmsd_threshold}å']],
        on='protein', 
        suffixes=(f'_{method1}', f'_{method2}')
    )
    
    # Identify cases where one method succeeds and the other fails
    method1_only_success = merged[
        merged[f'rmsd_≤_{rmsd_threshold}å_{method1}'] & 
        ~merged[f'rmsd_≤_{rmsd_threshold}å_{method2}']
    ]
    
    method2_only_success = merged[
        ~merged[f'rmsd_≤_{rmsd_threshold}å_{method1}'] & 
        merged[f'rmsd_≤_{rmsd_threshold}å_{method2}']
    ]
    
    both_success = merged[
        merged[f'rmsd_≤_{rmsd_threshold}å_{method1}'] & 
        merged[f'rmsd_≤_{rmsd_threshold}å_{method2}']
    ]
    
    both_fail = merged[
        ~merged[f'rmsd_≤_{rmsd_threshold}å_{method1}'] & 
        ~merged[f'rmsd_≤_{rmsd_threshold}å_{method2}']
    ]
    
    # Calculate statistics
    total_proteins = len(merged)
    success_rate1 = best_rmsd1[f'rmsd_≤_{rmsd_threshold}å'].mean() * 100
    success_rate2 = best_rmsd2[f'rmsd_≤_{rmsd_threshold}å'].mean() * 100
    
    stats = {
        'total_proteins': total_proteins,
        f'{method1}_success_rate': success_rate1,
        f'{method2}_success_rate': success_rate2,
        'both_success_count': len(both_success),
        'both_success_percent': len(both_success) / total_proteins * 100,
        f'{method1}_only_success_count': len(method1_only_success),
        f'{method1}_only_success_percent': len(method1_only_success) / total_proteins * 100,
        f'{method2}_only_success_count': len(method2_only_success),
        f'{method2}_only_success_percent': len(method2_only_success) / total_proteins * 100,
        'both_fail_count': len(both_fail),
        'both_fail_percent': len(both_fail) / total_proteins * 100,
    }
    
    return {
        'stats': stats,
        f'{method1}_only_success': method1_only_success,
        f'{method2}_only_success': method2_only_success,
        'both_success': both_success,
        'both_fail': both_fail,
        'merged': merged
    }

In [ ]:
compare_method_performance(
    df_combined, 
    method1='icm', 
    method2='diffdock_pocket_only', 
    rmsd_threshold=2.0
)

In [ ]:
compare_method_performance(
    df_combined, 
    method1='icm', 
    method2='surfdock', 
    rmsd_threshold=2.0
)

In [ ]:
compare_method_performance(
    df_combined, 
    method1='icm', 
    method2='gnina', 
    rmsd_threshold=2.0
)

In [ ]:
compare_method_performance(
    df_combined, 
    method1='icm', 
    method2='chai-1', 
    rmsd_threshold=2.0
)

In [ ]:
def analyze_method_differences(df_combined, method1, method2, rmsd_threshold=2.0, annotation_df=None, inputs_df=None, posebusters_df=None):
    """
    Analyze differences between two docking methods by examining the properties of proteins
    where one method succeeds and the other fails.
    
    Parameters:
    -----------
    df_combined : pandas.DataFrame
        Combined dataframe with results from all methods
    method1 : str
        Name of the first method to compare
    method2 : str
        Name of the second method to compare
    rmsd_threshold : float, optional
        RMSD threshold for considering a docking successful, default is 2.0 Å
    annotation_df : pandas.DataFrame, optional
        Dataframe containing protein annotations
    inputs_df : pandas.DataFrame, optional
        Additional annotation dataframe (e.g., containing ion information)
    posebusters_df : pandas.DataFrame, optional
        PoseBusters-specific annotations for validity checks
        
    Returns:
    --------
    dict
        Dictionary containing analysis results and visualizations
    """
    import matplotlib.pyplot as plt
    import numpy as np
    import seaborn as sns
    import scipy.stats as stats
    import ast
    
    # First get the comparison results
    comparison = compare_method_performance(df_combined, method1, method2, rmsd_threshold)
    
    # Extract success and failure cases
    method1_only = comparison[f'{method1}_only_success']
    method2_only = comparison[f'{method2}_only_success']
    
    print(f"Total proteins: {comparison['stats']['total_proteins']}")
    print(f"{method1} success rate: {comparison['stats'][f'{method1}_success_rate']:.2f}%")
    print(f"{method2} success rate: {comparison['stats'][f'{method2}_success_rate']:.2f}%")
    print(f"Both methods successful: {comparison['stats']['both_success_count']} ({comparison['stats']['both_success_percent']:.2f}%)")
    print(f"{method1} only successful: {comparison['stats'][f'{method1}_only_success_count']} ({comparison['stats'][f'{method1}_only_success_percent']:.2f}%)")
    print(f"{method2} only successful: {comparison['stats'][f'{method2}_only_success_count']} ({comparison['stats'][f'{method2}_only_success_percent']:.2f}%)")
    print(f"Both methods failed: {comparison['stats']['both_fail_count']} ({comparison['stats']['both_fail_percent']:.2f}%)")
    
    # If no annotation data is available, return basic comparison
    if annotation_df is None and posebusters_df is None:
        return comparison
    
    # Create merged dataframes with annotations
    method1_proteins = method1_only['protein'].tolist()
    method2_proteins = method2_only['protein'].tolist()
    
    # Get annotations for method-specific successes
    method1_annot = method1_only.copy()
    method2_annot = method2_only.copy()
    
    # Merge with standard annotations if available
    if annotation_df is not None:
        method1_annot = pd.merge(
            method1_annot,
            annotation_df.drop_duplicates(subset=['system_id']),
            left_on='protein',
            right_on='system_id',
            how='left'
        )
        
        method2_annot = pd.merge(
            method2_annot,
            annotation_df.drop_duplicates(subset=['system_id']),
            left_on='protein',
            right_on='system_id',
            how='left'
        )
    
    # Merge with additional inputs if available
    if inputs_df is not None:
        # Identify common columns to avoid duplicates
        intersection_columns = set(inputs_df.columns).intersection(set(annotation_df.columns)) if annotation_df is not None else set()
        intersection_columns.remove('system_id') if 'system_id' in intersection_columns else None
        
        method1_annot = pd.merge(
            method1_annot,
            inputs_df.drop(columns=list(intersection_columns)) if intersection_columns else inputs_df,
            left_on='protein',
            right_on='system_id',
            how='left'
        )
        
        method2_annot = pd.merge(
            method2_annot,
            inputs_df.drop(columns=list(intersection_columns)) if intersection_columns else inputs_df,
            left_on='protein',
            right_on='system_id',
            how='left'
        )
        
        # Process ion_paths if it exists
        if 'ion_paths' in method1_annot.columns:
            method1_annot['ion_paths'] = method1_annot['ion_paths'].apply(
                lambda x: ast.literal_eval(x) if isinstance(x, str) else x
            )
            method1_annot['has_ion'] = method1_annot['ion_paths'].apply(
                lambda x: len(x) > 0 if isinstance(x, list) else False
            )
        
        if 'ion_paths' in method2_annot.columns:
            method2_annot['ion_paths'] = method2_annot['ion_paths'].apply(
                lambda x: ast.literal_eval(x) if isinstance(x, str) else x
            )
            method2_annot['has_ion'] = method2_annot['ion_paths'].apply(
                lambda x: len(x) > 0 if isinstance(x, list) else False
            )
    
    # Merge with PoseBusters data if available
    if posebusters_df is not None:
        method1_annot = pd.merge(
            method1_annot,
            posebusters_df,
            left_on=['protein', 'method'],
            right_on=['protein', 'method'],
            how='left'
        )
        
        method2_annot = pd.merge(
            method2_annot,
            posebusters_df,
            left_on=['protein', 'method'],
            right_on=['protein', 'method'],
            how='left'
        )
    
    # Analysis begins here - start with key properties that might differ
    results = {}
    
    # Define PoseBusters-specific columns
    posebusters_columns = [
        # chemical validity and consistency
        "mol_pred_loaded", "mol_true_loaded", "mol_cond_loaded", "sanitization",
        "molecular_formula", "molecular_bonds", "tetrahedral_chirality", 
        "double_bond_stereochemistry",
        # intramolecular validity
        "bond_lengths", "bond_angles", "internal_steric_clash",
        "aromatic_ring_flatness", "double_bond_flatness", "internal_energy",
        # intermolecular validity
        "minimum_distance_to_protein", "minimum_distance_to_organic_cofactors",
        "minimum_distance_to_inorganic_cofactors", "volume_overlap_with_protein",
        "volume_overlap_with_organic_cofactors", "volume_overlap_with_inorganic_cofactors"
    ]
    
    # Identify key numerical properties to analyze
    numerical_properties = [
        'entry_resolution', 
        'entry_validation_molprobity',
        'system_num_pocket_residues',
        'system_num_interactions',
        'ligand_molecular_weight',
        'ligand_crippen_clogp',
        'ligand_num_hba',
        'ligand_num_hbd',
        'ligand_num_rings',
        'ligand_num_interacting_residues',
        'system_num_protein_chains',
        'minimum_distance_to_protein',
        'internal_energy'
    ]
    
    # Boolean properties to analyze
    boolean_properties = [
        'has_ion',
        'ligand_is_covalent',
        'ligand_is_cofactor',
        'ligand_is_artifact',
        'molecular_formula',
        'molecular_bonds',
        'tetrahedral_chirality',
        'double_bond_stereochemistry',
        'bond_lengths',
        'bond_angles',
        'internal_steric_clash',
        'sanitization'
    ]
    
    # Create a figure to visualize numerical property distributions
    num_numerical = sum(1 for prop in numerical_properties if prop in method1_annot.columns)
    if num_numerical > 0:
        fig, axes = plt.subplots(
            (num_numerical + 2) // 3, 3, 
            figsize=(18, 5 * ((num_numerical + 2) // 3)), 
            constrained_layout=True
        )
        axes = axes.flatten() if hasattr(axes, 'flatten') else [axes]
    
    # Compare numerical properties
    ax_idx = 0
    numerical_results = {}
    
    for prop in numerical_properties:
        if prop not in method1_annot.columns or prop not in method2_annot.columns:
            continue
            
        # Get clean data
        m1_data = method1_annot[prop].dropna()
        m2_data = method2_annot[prop].dropna()
        
        if len(m1_data) < 3 or len(m2_data) < 3:
            # Not enough data for statistical comparison
            continue
        
        # Compute statistics
        m1_mean = m1_data.mean()
        m2_mean = m2_data.mean()
        m1_median = m1_data.median()
        m2_median = m2_data.median()
        
        # Perform statistical test (Mann-Whitney U test - non-parametric)
        try:
            stat, p_value = stats.mannwhitneyu(m1_data, m2_data, alternative='two-sided')
            significant = p_value < 0.05
        except ValueError:
            # Handle case where test cannot be performed
            stat, p_value, significant = None, None, False
        
        numerical_results[prop] = {
            f'{method1}_mean': m1_mean,
            f'{method2}_mean': m2_mean,
            f'{method1}_median': m1_median,
            f'{method2}_median': m2_median,
            'p_value': p_value,
            'significant': significant
        }
        
        # Create visualization
        if num_numerical > 0 and ax_idx < len(axes):
            sns.histplot(m1_data, ax=axes[ax_idx], color='blue', alpha=0.5, 
                         kde=True, label=f"{method1} only success")
            sns.histplot(m2_data, ax=axes[ax_idx], color='orange', alpha=0.5, 
                         kde=True, label=f"{method2} only success")
            
            axes[ax_idx].axvline(m1_mean, color='blue', linestyle='--', 
                                 label=f"{method1} mean: {m1_mean:.2f}")
            axes[ax_idx].axvline(m2_mean, color='orange', linestyle='--', 
                                 label=f"{method2} mean: {m2_mean:.2f}")
            
            axes[ax_idx].set_title(f"{prop} Distribution\n" + 
                                 (f"p={p_value:.4f} (significant)" if significant else 
                                  f"p={p_value:.4f} (not significant)" if p_value else "No statistical test"))
            axes[ax_idx].set_xlabel(prop)
            axes[ax_idx].set_ylabel("Frequency")
            axes[ax_idx].legend()
            
            ax_idx += 1
    
    # Fill remaining axes if any
    if num_numerical > 0:
        for i in range(ax_idx, len(axes)):
            axes[i].axis('off')
        
        plt.suptitle(f"Comparison of Properties: {method1} vs {method2} Unique Successes", fontsize=16)
        plt.tight_layout()
        plt.show()
    
    # Compare boolean properties
    boolean_results = {}
    
    for prop in boolean_properties:
        if prop not in method1_annot.columns or prop not in method2_annot.columns:
            continue
        
        # Calculate proportions
        m1_prop = method1_annot[prop].mean() * 100  # percentage
        m2_prop = method2_annot[prop].mean() * 100  # percentage
        
        # Fisher's exact test for categorical data
        try:
            contingency = [
                [method1_annot[prop].sum(), len(method1_annot) - method1_annot[prop].sum()],
                [method2_annot[prop].sum(), len(method2_annot) - method2_annot[prop].sum()]
            ]
            
            _, p_value = stats.fisher_exact(contingency)
            significant = p_value < 0.05
        except:
            p_value, significant = None, False
        
        boolean_results[prop] = {
            f'{method1}_percentage': m1_prop,
            f'{method2}_percentage': m2_prop,
            'difference': m1_prop - m2_prop,
            'p_value': p_value,
            'significant': significant
        }
    
    # Visualize boolean properties
    if boolean_results:
        plt.figure(figsize=(12, 6))
        props = list(boolean_results.keys())
        m1_props = [boolean_results[p][f'{method1}_percentage'] for p in props]
        m2_props = [boolean_results[p][f'{method2}_percentage'] for p in props]
        
        x = np.arange(len(props))
        width = 0.35
        
        bars1 = plt.bar(x - width/2, m1_props, width, label=f'{method1} only success', color='blue', alpha=0.7)
        bars2 = plt.bar(x + width/2, m2_props, width, label=f'{method2} only success', color='orange', alpha=0.7)
        
        # Add significance markers
        for i, prop in enumerate(props):
            if boolean_results[prop]['significant']:
                plt.text(i, max(m1_props[i], m2_props[i]) + 3, '*', 
                         ha='center', fontsize=14, fontweight='bold')
        
        plt.xlabel('Property')
        plt.ylabel('Percentage (%)')
        plt.title(f'Comparison of Binary Properties: {method1} vs {method2} Unique Successes')
        plt.xticks(x, props, rotation=45, ha='right')
        plt.legend()
        plt.grid(axis='y', alpha=0.3)
        
        # Add data labels
        for i, v in enumerate(m1_props):
            plt.text(i - width/2, v + 1, f'{v:.1f}%', ha='center')
        for i, v in enumerate(m2_props):
            plt.text(i + width/2, v + 1, f'{v:.1f}%', ha='center')
            
        plt.tight_layout()
        plt.show()
    
    # Summary table for numerical properties with significant differences
    significant_numerical = {k: v for k, v in numerical_results.items() if v.get('significant', False)}
    if significant_numerical:
        print("\nSignificant differences in numerical properties:")
        for prop, stats in significant_numerical.items():
            print(f"{prop}: {method1} mean = {stats[f'{method1}_mean']:.2f}, "
                  f"{method2} mean = {stats[f'{method2}_mean']:.2f}, "
                  f"p-value = {stats['p_value']:.4f}")
    
    # Summary table for boolean properties with significant differences
    significant_boolean = {k: v for k, v in boolean_results.items() if v.get('significant', False)}
    if significant_boolean:
        print("\nSignificant differences in boolean properties:")
        for prop, stats in significant_boolean.items():
            print(f"{prop}: {method1} = {stats[f'{method1}_percentage']:.1f}%, "
                  f"{method2} = {stats[f'{method2}_percentage']:.1f}%, "
                  f"p-value = {stats['p_value']:.4f}")
    
    # Conclusions
    print("\nConclusions:")
    if significant_numerical or significant_boolean:
        print(f"Based on the analysis, {method1} and {method2} show different performance patterns:")
        
        for prop, stats in significant_numerical.items():
            if stats[f'{method1}_mean'] > stats[f'{method2}_mean']:
                print(f"- {method1} performs better on proteins with higher {prop}")
            else:
                print(f"- {method2} performs better on proteins with higher {prop}")
                
        for prop, stats in significant_boolean.items():
            if stats[f'{method1}_percentage'] > stats[f'{method2}_percentage']:
                print(f"- {method1} performs better on proteins {'with' if 'has_' in prop or 'is_' in prop else 'that pass'} {prop}")
            else:
                print(f"- {method2} performs better on proteins {'with' if 'has_' in prop or 'is_' in prop else 'that pass'} {prop}")
    else:
        print(f"The analysis did not reveal statistically significant differences in the properties of proteins where {method1} or {method2} uniquely succeed.")
    
    # Return compiled results
    return {
        'comparison': comparison,
        'numerical_properties': numerical_results,
        'boolean_properties': boolean_results,
        f'{method1}_annotations': method1_annot,
        f'{method2}_annotations': method2_annot
    }

In [ ]:
# Let's analyze the differences between ICM and DiffDock methods
icm_diffdock_analysis = analyze_method_differences(
    df_combined, 
    method1='icm', 
    method2='diffdock', 
    rmsd_threshold=2.0, 
    annotation_df=test_annot_df, 
    inputs_df=inputs_csv
)

# We can also compare ICM with Gnina
icm_gnina_analysis = analyze_method_differences(
    df_combined, 
    method1='icm', 
    method2='gnina', 
    rmsd_threshold=2.0, 
    annotation_df=test_annot_df, 
    inputs_df=inputs_csv
)

# And compare DiffDock with SurfDock
diffdock_surfdock_analysis = analyze_method_differences(
    df_combined, 
    method1='diffdock', 
    method2='surfdock', 
    rmsd_threshold=2.0, 
    annotation_df=test_annot_df, 
    inputs_df=inputs_csv
)

In [ ]:
# Compare methods with full analysis
posebusters_results = None
analysis_results = analyze_method_differences(
    df_combined, 
    method1='icm', 
    method2='diffdock_pocket_only', 
    rmsd_threshold=2.0, 
    annotation_df=test_annot_df, 
    inputs_df=inputs_csv,
    posebusters_df=posebusters_results
)

# Examine specific findings
significant_numerical = {k: v for k, v in analysis_results['numerical_properties'].items() 
                         if v.get('significant', False)}
significant_boolean = {k: v for k, v in analysis_results['boolean_properties'].items() 
                       if v.get('significant', False)}

# Get list of proteins where method1 succeeds but method2 fails
method1_only_proteins = analysis_results[f'icm_annotations']['protein'].tolist()

In [ ]:
analysis_results = analyze_method_differences(
    df_combined, 
    method1='icm', 
    method2='surfdock', 
    rmsd_threshold=2.0, 
    annotation_df=test_annot_df, 
    inputs_df=inputs_csv,
    posebusters_df=posebusters_results
)

# Examine specific findings
significant_numerical = {k: v for k, v in analysis_results['numerical_properties'].items() 
                         if v.get('significant', False)}
significant_boolean = {k: v for k, v in analysis_results['boolean_properties'].items() 
                       if v.get('significant', False)}

# Get list of proteins where method1 succeeds but method2 fails
method1_only_proteins = analysis_results[f'icm_annotations']['protein'].tolist()

In [ ]:
analysis_results = analyze_method_differences(
    df_combined, 
    method1='surfdock', 
    method2='gnina', 
    rmsd_threshold=2.0, 
    annotation_df=test_annot_df, 
    inputs_df=inputs_csv,
    posebusters_df=posebusters_results
)

# Examine specific findings
significant_numerical = {k: v for k, v in analysis_results['numerical_properties'].items() 
                         if v.get('significant', False)}
significant_boolean = {k: v for k, v in analysis_results['boolean_properties'].items() 
                       if v.get('significant', False)}

# Get list of proteins where method1 succeeds but method2 fails
method1_only_proteins = analysis_results[f'surfdock_annotations']['protein'].tolist()

In [ ]:
def analyze_multiple_methods(df_combined, methods, rmsd_threshold=2.0, annotation_df=None, inputs_df=None):
    """
    Analyze patterns in proteins where all specified methods either succeed or fail.
    
    Parameters:
    -----------
    df_combined : pandas.DataFrame
        Combined dataframe with results from all methods
    methods : list
        List of method names to analyze
    rmsd_threshold : float, optional
        RMSD threshold for considering a docking successful, default is 2.0 Å
    annotation_df : pandas.DataFrame, optional
        Dataframe containing protein annotations
    inputs_df : pandas.DataFrame, optional
        Additional annotation dataframe (e.g., containing ion information)
        
    Returns:
    --------
    dict
        Dictionary containing analysis results and visualizations
    """
    import matplotlib.pyplot as plt
    import numpy as np
    import seaborn as sns
    from scipy import stats
    import ast
    import pandas as pd
    
    # Get best RMSD per protein for each method
    best_rmsd_by_method = {}
    for method in methods:
        df_method = df_combined[df_combined['method'] == method]
        best_rmsd = df_method.groupby('protein').apply(
            lambda x: x.loc[x['rmsd'].idxmin()] if not x['rmsd'].isna().all() else x.iloc[0]
        ).reset_index(drop=True)
        
        # Create success column
        best_rmsd[f'rmsd_≤_{rmsd_threshold}å'] = best_rmsd['rmsd'] <= rmsd_threshold
        
        best_rmsd_by_method[method] = best_rmsd
    
    # Find common proteins across all methods
    common_proteins = set(best_rmsd_by_method[methods[0]]['protein'])
    for method in methods[1:]:
        common_proteins = common_proteins.intersection(set(best_rmsd_by_method[method]['protein']))
    common_proteins = list(common_proteins)
    
    # Create a dataframe to track success/failure for each protein across all methods
    success_matrix = pd.DataFrame({'protein': common_proteins})
    for method in methods:
        method_success = {}
        for protein in common_proteins:
            success = best_rmsd_by_method[method][
                best_rmsd_by_method[method]['protein'] == protein
            ][f'rmsd_≤_{rmsd_threshold}å'].iloc[0]
            method_success[protein] = success
        success_matrix[f'{method}_success'] = success_matrix['protein'].map(method_success)
    
    # Calculate total successes for each protein
    success_matrix['total_successes'] = success_matrix[[f'{method}_success' for method in methods]].sum(axis=1)
    
    # Identify proteins where all methods succeed or all methods fail
    all_success_proteins = success_matrix[success_matrix['total_successes'] == len(methods)]['protein'].tolist()
    all_failure_proteins = success_matrix[success_matrix['total_successes'] == 0]['protein'].tolist()
    
    # Calculate statistics
    total_proteins = len(common_proteins)
    all_success_count = len(all_success_proteins)
    all_failure_count = len(all_failure_proteins)
    all_success_percent = (all_success_count / total_proteins) * 100 if total_proteins > 0 else 0
    all_failure_percent = (all_failure_count / total_proteins) * 100 if total_proteins > 0 else 0
    
    # Print basic statistics
    print(f"Total proteins evaluated across all {len(methods)} methods: {total_proteins}")
    print(f"Proteins where all methods succeed: {all_success_count} ({all_success_percent:.2f}%)")
    print(f"Proteins where all methods fail: {all_failure_count} ({all_failure_percent:.2f}%)")
    
    # Create dataframes for analysis
    all_success_df = pd.DataFrame({'protein': all_success_proteins})
    all_failure_df = pd.DataFrame({'protein': all_failure_proteins})
    
    # If no annotation data is available, return basic statistics
    if annotation_df is None and inputs_df is None:
        return {
            'all_success_proteins': all_success_proteins,
            'all_failure_proteins': all_failure_proteins,
            'success_matrix': success_matrix,
            'statistics': {
                'total_proteins': total_proteins,
                'all_success_count': all_success_count,
                'all_failure_count': all_failure_count,
                'all_success_percent': all_success_percent,
                'all_failure_percent': all_failure_percent
            }
        }
    
    # Merge with annotation data
    if annotation_df is not None:
        all_success_df = pd.merge(
            all_success_df,
            annotation_df.drop_duplicates(subset=['system_id']),
            left_on='protein',
            right_on='system_id',
            how='left'
        )
        
        all_failure_df = pd.merge(
            all_failure_df,
            annotation_df.drop_duplicates(subset=['system_id']),
            left_on='protein',
            right_on='system_id',
            how='left'
        )
    
    # Merge with additional input data
    if inputs_df is not None:
        # Identify common columns to avoid duplicates
        intersection_columns = set(inputs_df.columns).intersection(set(annotation_df.columns)) if annotation_df is not None else set()
        intersection_columns.remove('system_id') if 'system_id' in intersection_columns else None
        
        all_success_df = pd.merge(
            all_success_df,
            inputs_df.drop(columns=list(intersection_columns)) if intersection_columns else inputs_df,
            left_on='protein',
            right_on='system_id',
            how='left'
        )
        
        all_failure_df = pd.merge(
            all_failure_df,
            inputs_df.drop(columns=list(intersection_columns)) if intersection_columns else inputs_df,
            left_on='protein',
            right_on='system_id',
            how='left'
        )
        
        # Process ion_paths if it exists
        if 'ion_paths' in all_success_df.columns:
            all_success_df['ion_paths'] = all_success_df['ion_paths'].apply(
                lambda x: ast.literal_eval(x) if isinstance(x, str) else x
            )
            all_success_df['has_ion'] = all_success_df['ion_paths'].apply(
                lambda x: len(x) > 0 if isinstance(x, list) else False
            )
        
        if 'ion_paths' in all_failure_df.columns:
            all_failure_df['ion_paths'] = all_failure_df['ion_paths'].apply(
                lambda x: ast.literal_eval(x) if isinstance(x, str) else x
            )
            all_failure_df['has_ion'] = all_failure_df['ion_paths'].apply(
                lambda x: len(x) > 0 if isinstance(x, list) else False
            )
    
    # Define properties to analyze
    posebusters_columns = [
        # chemical validity and consistency
        "mol_pred_loaded", "mol_true_loaded", "mol_cond_loaded", "sanitization",
        "molecular_formula", "molecular_bonds", "tetrahedral_chirality", 
        "double_bond_stereochemistry",
        # intramolecular validity
        "bond_lengths", "bond_angles", "internal_steric_clash",
        "aromatic_ring_flatness", "double_bond_flatness", "internal_energy",
        # intermolecular validity
        "minimum_distance_to_protein", "minimum_distance_to_organic_cofactors",
        "minimum_distance_to_inorganic_cofactors", "volume_overlap_with_protein",
        "volume_overlap_with_organic_cofactors", "volume_overlap_with_inorganic_cofactors"
    ]
    
    # Identify key numerical properties to analyze
    numerical_properties = [
        'entry_resolution', 
        'entry_validation_molprobity',
        'system_num_pocket_residues',
        'system_num_interactions',
        'ligand_molecular_weight',
        'ligand_crippen_clogp',
        'ligand_num_hba',
        'ligand_num_hbd',
        'ligand_num_rings',
        'ligand_num_interacting_residues',
        'system_num_protein_chains',
        'minimum_distance_to_protein',
        'internal_energy'
    ]
    
    # Boolean properties to analyze
    boolean_properties = [
        'has_ion',
        'ligand_is_covalent',
        'ligand_is_cofactor',
        'ligand_is_artifact',
        'molecular_formula',
        'molecular_bonds',
        'tetrahedral_chirality',
        'double_bond_stereochemistry',
        'bond_lengths',
        'bond_angles',
        'internal_steric_clash',
        'sanitization'
    ]
    
    # Create a figure to visualize numerical property distributions
    num_numerical = sum(1 for prop in numerical_properties 
                      if prop in all_success_df.columns and prop in all_failure_df.columns)
    
    if num_numerical > 0:
        fig, axes = plt.subplots(
            (num_numerical + 2) // 3, 3, 
            figsize=(18, 5 * ((num_numerical + 2) // 3)), 
            constrained_layout=True
        )
        axes = axes.flatten() if hasattr(axes, 'flatten') else [axes]
        
    # Compare numerical properties
    ax_idx = 0
    numerical_results = {}
    
    for prop in numerical_properties:
        if (prop not in all_success_df.columns or prop not in all_failure_df.columns or 
            all_success_df[prop].isna().all() or all_failure_df[prop].isna().all()):
            continue
            
        # Get clean data
        success_data = all_success_df[prop].dropna()
        failure_data = all_failure_df[prop].dropna()
        
        if len(success_data) < 3 or len(failure_data) < 3:
            # Not enough data for statistical comparison
            continue
        
        # Compute statistics
        success_mean = success_data.mean()
        failure_mean = failure_data.mean()
        success_median = success_data.median()
        failure_median = failure_data.median()
        
        # Perform statistical test (Mann-Whitney U test - non-parametric)
        try:
            stat, p_value = stats.mannwhitneyu(success_data, failure_data, alternative='two-sided')
            significant = p_value < 0.05
        except ValueError:
            # Handle case where test cannot be performed
            stat, p_value, significant = None, None, False
        
        numerical_results[prop] = {
            'success_mean': success_mean,
            'failure_mean': failure_mean,
            'success_median': success_median,
            'failure_median': failure_median,
            'p_value': p_value,
            'significant': significant
        }
        
        # Create visualization
        if num_numerical > 0 and ax_idx < len(axes):
            sns.histplot(success_data, ax=axes[ax_idx], color='green', alpha=0.5, 
                         kde=True, label="All methods succeed")
            sns.histplot(failure_data, ax=axes[ax_idx], color='red', alpha=0.5, 
                         kde=True, label="All methods fail")
            
            axes[ax_idx].axvline(success_mean, color='green', linestyle='--', 
                                 label=f"Success mean: {success_mean:.2f}")
            axes[ax_idx].axvline(failure_mean, color='red', linestyle='--', 
                                 label=f"Failure mean: {failure_mean:.2f}")
            
            axes[ax_idx].set_title(f"{prop} Distribution\n" + 
                                 (f"p={p_value:.4f} (significant)" if significant else 
                                  f"p={p_value:.4f} (not significant)" if p_value else "No statistical test"))
            axes[ax_idx].set_xlabel(prop)
            axes[ax_idx].set_ylabel("Frequency")
            axes[ax_idx].legend()
            
            ax_idx += 1
    
    # Fill remaining axes if any
    if num_numerical > 0:
        for i in range(ax_idx, len(axes)):
            axes[i].axis('off')
        
        plt.suptitle(f"Comparison of Properties: Success vs Failure Cases Across All Methods", fontsize=16)
        plt.tight_layout()
        plt.show()
    
    # Compare boolean properties
    boolean_results = {}
    
    for prop in boolean_properties:
        if (prop not in all_success_df.columns or prop not in all_failure_df.columns or 
            all_success_df[prop].isna().all() or all_failure_df[prop].isna().all()):
            continue
        
        # Calculate proportions
        success_prop = all_success_df[prop].mean() * 100  # percentage
        failure_prop = all_failure_df[prop].mean() * 100  # percentage
        
        # Fisher's exact test for categorical data
        try:
            contingency = [
                [all_success_df[prop].sum(), len(all_success_df) - all_success_df[prop].sum()],
                [all_failure_df[prop].sum(), len(all_failure_df) - all_failure_df[prop].sum()]
            ]
            
            _, p_value = stats.fisher_exact(contingency)
            significant = p_value < 0.05
        except:
            p_value, significant = None, False
        
        boolean_results[prop] = {
            'success_percentage': success_prop,
            'failure_percentage': failure_prop,
            'difference': success_prop - failure_prop,
            'p_value': p_value,
            'significant': significant
        }
    
    # Visualize boolean properties
    if boolean_results:
        plt.figure(figsize=(12, 6))
        props = list(boolean_results.keys())
        success_props = [boolean_results[p]['success_percentage'] for p in props]
        failure_props = [boolean_results[p]['failure_percentage'] for p in props]
        
        x = np.arange(len(props))
        width = 0.35
        
        bars1 = plt.bar(x - width/2, success_props, width, label='All methods succeed', color='green', alpha=0.7)
        bars2 = plt.bar(x + width/2, failure_props, width, label='All methods fail', color='red', alpha=0.7)
        
        # Add significance markers
        for i, prop in enumerate(props):
            if boolean_results[prop]['significant']:
                plt.text(i, max(success_props[i], failure_props[i]) + 3, '*', 
                         ha='center', fontsize=14, fontweight='bold')
        
        plt.xlabel('Property')
        plt.ylabel('Percentage (%)')
        plt.title('Comparison of Binary Properties: Success vs Failure Cases')
        plt.xticks(x, props, rotation=45, ha='right')
        plt.legend()
        plt.grid(axis='y', alpha=0.3)
        
        # Add data labels
        for i, v in enumerate(success_props):
            plt.text(i - width/2, v + 1, f'{v:.1f}%', ha='center')
        for i, v in enumerate(failure_props):
            plt.text(i + width/2, v + 1, f'{v:.1f}%', ha='center')
            
        plt.tight_layout()
        plt.show()
    
    # Summary table for numerical properties with significant differences
    significant_numerical = {k: v for k, v in numerical_results.items() if v.get('significant', False)}
    if significant_numerical:
        print("\nSignificant differences in numerical properties:")
        for prop, stats in significant_numerical.items():
            print(f"{prop}: Success mean = {stats['success_mean']:.2f}, "
                  f"Failure mean = {stats['failure_mean']:.2f}, "
                  f"p-value = {stats['p_value']:.4f}")
    
    # Summary table for boolean properties with significant differences
    significant_boolean = {k: v for k, v in boolean_results.items() if v.get('significant', False)}
    if significant_boolean:
        print("\nSignificant differences in boolean properties:")
        for prop, stats in significant_boolean.items():
            print(f"{prop}: Success = {stats['success_percentage']:.1f}%, "
                  f"Failure = {stats['failure_percentage']:.1f}%, "
                  f"p-value = {stats['p_value']:.4f}")
    
    # Conclusions
    print("\nConclusions:")
    if significant_numerical or significant_boolean:
        print("Based on the analysis, key differences between cases where all methods succeed vs. fail:")
        
        for prop, stats in significant_numerical.items():
            if stats['success_mean'] > stats['failure_mean']:
                print(f"- Higher {prop} is associated with successful docking across all methods")
            else:
                print(f"- Lower {prop} is associated with successful docking across all methods")
                
        for prop, stats in significant_boolean.items():
            if stats['success_percentage'] > stats['failure_percentage']:
                print(f"- {'Presence of' if 'has_' in prop or 'is_' in prop else 'Passing'} {prop} is associated with successful docking")
            else:
                print(f"- {'Absence of' if 'has_' in prop or 'is_' in prop else 'Failing'} {prop} is associated with successful docking")
    else:
        print("The analysis did not reveal statistically significant differences in the properties of proteins where all methods succeed vs. fail.")
    
    # Return compiled results
    return {
        'all_success_proteins': all_success_proteins,
        'all_failure_proteins': all_failure_proteins,
        'success_matrix': success_matrix,
        'numerical_properties': numerical_results,
        'boolean_properties': boolean_results,
        'success_df': all_success_df,
        'failure_df': all_failure_df,
        'statistics': {
            'total_proteins': total_proteins,
            'all_success_count': all_success_count,
            'all_failure_count': all_failure_count,
            'all_success_percent': all_success_percent,
            'all_failure_percent': all_failure_percent
        }
    }

In [ ]:
# Analyze multiple methods
methods_to_analyze = ['icm', 'diffdock', 'gnina', 'surfdock']

multi_method_analysis = analyze_multiple_methods(
    df_combined, 
    methods=methods_to_analyze,
    rmsd_threshold=2.0, 
    annotation_df=test_annot_df, 
    inputs_df=inputs_csv
)

# Get proteins where all methods succeed or fail
universal_success_proteins = multi_method_analysis['all_success_proteins']
universal_failure_proteins = multi_method_analysis['all_failure_proteins']

# Check significant property differences
significant_numerical = {k: v for k, v in multi_method_analysis['numerical_properties'].items() 
                         if v.get('significant', False)}
significant_boolean = {k: v for k, v in multi_method_analysis['boolean_properties'].items() 
                       if v.get('significant', False)}

In [ ]:
len(universal_success_proteins), len(universal_failure_proteins)

## Analysis of the protein-ligand annotations

In [ ]:
def analyze_universal_properties(df_combined, methods, annotation_df=None, inputs_df=None, rmsd_threshold=2.0):
    """
    Analyze protein and ligand properties to identify factors associated with 
    universal success or failure across multiple docking methods.
    
    Parameters:
    -----------
    df_combined : pandas.DataFrame
        Combined dataframe with results from all methods
    methods : list
        List of method names to analyze
    annotation_df : pandas.DataFrame, optional
        Dataframe containing protein/ligand annotations
    inputs_df : pandas.DataFrame, optional
        Additional annotation dataframe (e.g., containing ion information)
    rmsd_threshold : float, optional
        RMSD threshold for considering a docking successful, default is 2.0 Å
        
    Returns:
    --------
    dict
        Dictionary containing analysis results and visualizations
    """
    import matplotlib.pyplot as plt
    import numpy as np
    import seaborn as sns
    import scipy.stats as stats
    import pandas as pd
    import ast
    from matplotlib.colors import LinearSegmentedColormap
    # Get best RMSD per protein for each method
    best_rmsd_by_method = {}
    for method in methods:
        df_method = df_combined[df_combined['method'] == method]
        best_rmsd = df_method.groupby('protein').apply(
            lambda x: x.loc[x['rmsd'].idxmin()] if not x['rmsd'].isna().all() else x.iloc[0]
        ).reset_index(drop=True)
        
        # Create success column
        best_rmsd[f'rmsd_≤_{rmsd_threshold}å'] = best_rmsd['rmsd'] <= rmsd_threshold
        
        best_rmsd_by_method[method] = best_rmsd
    
    # Find common proteins across all methods
    common_proteins = set(best_rmsd_by_method[methods[0]]['protein'])
    for method in methods[1:]:
        common_proteins = common_proteins.intersection(set(best_rmsd_by_method[method]['protein']))
    common_proteins = list(common_proteins)
    
    # Create a dataframe to track success/failure for each protein across all methods
    success_matrix = pd.DataFrame({'protein': common_proteins})
    for method in methods:
        method_success = {}
        for protein in common_proteins:
            success = best_rmsd_by_method[method][
                best_rmsd_by_method[method]['protein'] == protein
            ][f'rmsd_≤_{rmsd_threshold}å'].iloc[0]
            method_success[protein] = success
        success_matrix[f'{method}_success'] = success_matrix['protein'].map(method_success)
    
    # Calculate total successes for each protein
    success_matrix['total_successes'] = success_matrix[[f'{method}_success' for method in methods]].sum(axis=1)
    
    # Identify proteins where all methods succeed or all methods fail
    all_success_proteins = success_matrix[success_matrix['total_successes'] == len(methods)]['protein'].tolist()
    all_failure_proteins = success_matrix[success_matrix['total_successes'] == 0]['protein'].tolist()
    
    # Calculate statistics
    total_proteins = len(common_proteins)
    all_success_count = len(all_success_proteins)
    all_failure_count = len(all_failure_proteins)
    all_success_percent = (all_success_count / total_proteins) * 100 if total_proteins > 0 else 0
    all_failure_percent = (all_failure_count / total_proteins) * 100 if total_proteins > 0 else 0
    
    # Print basic statistics
    print(f"Total proteins evaluated across all {len(methods)} methods: {total_proteins}")
    print(f"Proteins where all methods succeed: {all_success_count} ({all_success_percent:.2f}%)")
    print(f"Proteins where all methods fail: {all_failure_count} ({all_failure_percent:.2f}%)")
    
    # Create dataframes for analysis
    all_success_df = pd.DataFrame({'protein': all_success_proteins})
    all_failure_df = pd.DataFrame({'protein': all_failure_proteins})
    
    # If no annotation data is available, return basic statistics
    if annotation_df is None and inputs_df is None:
        return {
            'all_success_proteins': all_success_proteins,
            'all_failure_proteins': all_failure_proteins,
            'success_matrix': success_matrix,
            'statistics': {
                'total_proteins': total_proteins,
                'all_success_count': all_success_count,
                'all_failure_count': all_failure_count,
                'all_success_percent': all_success_percent,
                'all_failure_percent': all_failure_percent
            }
        }
    
    # Merge with annotation data
    if annotation_df is not None:
        all_success_df = pd.merge(
            all_success_df,
            annotation_df.drop_duplicates(subset=['system_id']),
            left_on='protein',
            right_on='system_id',
            how='left'
        )
        
        all_failure_df = pd.merge(
            all_failure_df,
            annotation_df.drop_duplicates(subset=['system_id']),
            left_on='protein',
            right_on='system_id',
            how='left'
        )
    
    # Merge with additional input data
    if inputs_df is not None:
        # Identify common columns to avoid duplicates
        intersection_columns = set(inputs_df.columns).intersection(set(annotation_df.columns)) if annotation_df is not None else set()
        intersection_columns.remove('system_id') if 'system_id' in intersection_columns else None
        
        all_success_df = pd.merge(
            all_success_df,
            inputs_df.drop(columns=list(intersection_columns)) if intersection_columns else inputs_df,
            left_on='protein',
            right_on='system_id',
            how='left'
        )
        
        all_failure_df = pd.merge(
            all_failure_df,
            inputs_df.drop(columns=list(intersection_columns)) if intersection_columns else inputs_df,
            left_on='protein',
            right_on='system_id',
            how='left'
        )
        
        # Process ion_paths if it exists
        if 'ion_paths' in all_success_df.columns:
            all_success_df['ion_paths'] = all_success_df['ion_paths'].apply(
                lambda x: ast.literal_eval(x) if isinstance(x, str) else x
            )
            all_success_df['has_ion'] = all_success_df['ion_paths'].apply(
                lambda x: len(x) > 0 if isinstance(x, list) else False
            )
        
        if 'ion_paths' in all_failure_df.columns:
            all_failure_df['ion_paths'] = all_failure_df['ion_paths'].apply(
                lambda x: ast.literal_eval(x) if isinstance(x, str) else x
            )
            all_failure_df['has_ion'] = all_failure_df['ion_paths'].apply(
                lambda x: len(x) > 0 if isinstance(x, list) else False
            )
    
    # Define properties to analyze by type
    continuous_properties = [
        'entry_resolution', 
        'entry_validation_molprobity',
        'system_num_pocket_residues',
        'system_num_interactions',
        'ligand_molecular_weight',
        'ligand_crippen_clogp',
        'ligand_num_hba',
        'ligand_num_hbd',
        'ligand_num_rot_bonds',
        'ligand_num_interacting_residues',
        'ligand_num_neighboring_residues'
    ]
    
    discrete_properties = [
        'system_num_protein_chains',
        'ligand_num_rings'
    ]
    
    binary_properties = [
        'has_ion',
        'ligand_is_covalent',
        'ligand_is_cofactor',
        'ligand_is_artifact'
    ]
    
    # Results containers
    continuous_results = {}
    discrete_results = {}
    binary_results = {}
    
    # Analyze continuous properties
    # Count available continuous properties
    available_continuous = [p for p in continuous_properties 
                           if p in all_success_df.columns and p in all_failure_df.columns
                           and not all_success_df[p].isna().all() and not all_failure_df[p].isna().all()]
    
    if available_continuous:
        fig, axes = plt.subplots(
            (len(available_continuous) + 2) // 3, 3, 
            figsize=(18, 5 * ((len(available_continuous) + 2) // 3)), 
            constrained_layout=True
        )
        axes = axes.flatten() if hasattr(axes, 'flatten') else [axes]
        
        # Analyze each property
        for i, prop in enumerate(available_continuous):
            # Get clean data
            success_data = all_success_df[prop].dropna()
            failure_data = all_failure_df[prop].dropna()
            
            if len(success_data) < 3 or len(failure_data) < 3:
                # Not enough data for statistical comparison
                continue
            
            # Compute statistics
            success_mean = success_data.mean()
            failure_mean = failure_data.mean()
            success_median = success_data.median()
            failure_median = failure_data.median()
            
            # Perform statistical test (Mann-Whitney U test - non-parametric)
            try:
                stat, p_value = stats.mannwhitneyu(success_data, failure_data, alternative='two-sided')
                significant = p_value < 0.05
            except ValueError:
                # Handle case where test cannot be performed
                stat, p_value, significant = None, None, False
            
            continuous_results[prop] = {
                'success_mean': success_mean,
                'failure_mean': failure_mean,
                'success_median': success_median,
                'failure_median': failure_median,
                'p_value': p_value,
                'significant': significant
            }
            
            # Create visualization
            if i < len(axes):
                sns.histplot(success_data, ax=axes[i], color='green', alpha=0.5, 
                             kde=True, label="All methods succeed")
                sns.histplot(failure_data, ax=axes[i], color='red', alpha=0.5, 
                             kde=True, label="All methods fail")
                
                axes[i].axvline(success_mean, color='green', linestyle='--', 
                               label=f"Success mean: {success_mean:.2f}")
                axes[i].axvline(failure_mean, color='red', linestyle='--', 
                               label=f"Failure mean: {failure_mean:.2f}")
                
                axes[i].set_title(f"{prop} Distribution\n" + 
                                 (f"p={p_value:.4f} (significant)" if significant else 
                                  f"p={p_value:.4f} (not significant)" if p_value else "No statistical test"))
                axes[i].set_xlabel(prop)
                axes[i].set_ylabel("Frequency")
                axes[i].legend()
        
        # Hide empty subplots
        for i in range(len(available_continuous), len(axes)):
            axes[i].axis('off')
            
        plt.suptitle(f"Continuous Properties: Universal Success vs. Universal Failure", fontsize=16)
        plt.tight_layout()
        plt.show()
        
    # Analyze discrete properties
    available_discrete = [p for p in discrete_properties 
                         if p in all_success_df.columns and p in all_failure_df.columns
                         and not all_success_df[p].isna().all() and not all_failure_df[p].isna().all()]
    
    if available_discrete:
        fig, axes = plt.subplots(
            len(available_discrete), 2,
            figsize=(14, 5 * len(available_discrete)),
            constrained_layout=True
        )
        
        if len(available_discrete) == 1:
            axes = [axes]
            
        for i, prop in enumerate(available_discrete):
            # Get data distributions
            success_counts = all_success_df[prop].value_counts().sort_index()
            failure_counts = all_failure_df[prop].value_counts().sort_index()
            
            # Get all unique values
            all_values = sorted(set(success_counts.index) | set(failure_counts.index))
            
            # Fill missing values with 0
            for val in all_values:
                if val not in success_counts:
                    success_counts[val] = 0
                if val not in failure_counts:
                    failure_counts[val] = 0
                    
            success_counts = success_counts.sort_index()
            failure_counts = failure_counts.sort_index()
            
            # Convert to percentages for comparison
            success_pct = success_counts / success_counts.sum() * 100
            failure_pct = failure_counts / failure_counts.sum() * 100
            
            # Chi-square test
            try:
                contingency = np.zeros((2, len(all_values)))
                contingency[0, :] = success_counts.values
                contingency[1, :] = failure_counts.values
                chi2, p_value, dof, expected = stats.chi2_contingency(contingency)
                significant = p_value < 0.05
            except:
                chi2, p_value, significant = None, None, False
                
            discrete_results[prop] = {
                'success_distribution': success_counts,
                'failure_distribution': failure_counts,
                'success_percentage': success_pct,
                'failure_percentage': failure_pct,
                'chi2': chi2,
                'p_value': p_value,
                'significant': significant
            }
            
            # Create visualization - count plot
            axes[i][0].bar(success_pct.index.astype(str), success_pct.values, alpha=0.7, color='green', label='Success')
            axes[i][0].bar(failure_pct.index.astype(str), failure_pct.values, alpha=0.7, color='red', label='Failure')
            axes[i][0].set_title(f"{prop} Distribution (%)\n" + 
                              (f"p={p_value:.4f} (significant)" if significant else 
                               f"p={p_value:.4f} (not significant)" if p_value else "No statistical test"))
            axes[i][0].set_xlabel(prop)
            axes[i][0].set_ylabel("Percentage (%)")
            axes[i][0].legend()
            
            # Create visualization - success rate by value
            combined = pd.DataFrame({
                'value': list(success_counts.index) + list(failure_counts.index),
                'count': list(success_counts.values) + list(failure_counts.values),
                'type': ['success'] * len(success_counts) + ['failure'] * len(failure_counts)
            })
            
            pivot_df = combined.pivot_table(index='value', columns='type', values='count', fill_value=0)
            pivot_df['total'] = pivot_df['success'] + pivot_df['failure']
            pivot_df['success_rate'] = pivot_df['success'] / pivot_df['total'] * 100
            
            axes[i][1].bar(pivot_df.index.astype(str), pivot_df['success_rate'], color='blue')
            axes[i][1].set_title(f"Success Rate by {prop} Value")
            axes[i][1].set_xlabel(prop)
            axes[i][1].set_ylabel("Success Rate (%)")
            
            # Add data labels
            for j, v in enumerate(pivot_df['success_rate']):
                axes[i][1].text(j, v + 2, f'{v:.1f}%', ha='center')
                
            # Add count labels
            for j, v in enumerate(pivot_df['total']):
                axes[i][1].text(j, 5, f'n={int(v)}', ha='center')
                
        plt.suptitle(f"Discrete Properties: Universal Success vs. Universal Failure", fontsize=16)
        plt.tight_layout()
        plt.show()
    
    # Analyze binary properties
    available_binary = [p for p in binary_properties 
                       if p in all_success_df.columns and p in all_failure_df.columns
                       and not all_success_df[p].isna().all() and not all_failure_df[p].isna().all()]
    
    if available_binary:
        fig, axes = plt.subplots(
            (len(available_binary) + 1) // 2, 2,
            figsize=(14, 5 * ((len(available_binary) + 1) // 2)),
            constrained_layout=True
        )
        
        if len(available_binary) == 1:
            axes = [axes]
        else:
            axes = axes.flatten()
        
        for i, prop in enumerate(available_binary):
            # Convert to boolean if needed
            all_success_df[prop] = all_success_df[prop].astype(bool)
            all_failure_df[prop] = all_failure_df[prop].astype(bool)
            
            # Calculate proportions
            success_true = all_success_df[prop].mean() * 100
            failure_true = all_failure_df[prop].mean() * 100
            
            # Fisher's exact test
            try:
                contingency = [
                    [all_success_df[prop].sum(), len(all_success_df) - all_success_df[prop].sum()],
                    [all_failure_df[prop].sum(), len(all_failure_df) - all_failure_df[prop].sum()]
                ]
                
                _, p_value = stats.fisher_exact(contingency)
                significant = p_value < 0.05
            except:
                p_value, significant = None, False
                
            binary_results[prop] = {
                'success_true_pct': success_true,
                'failure_true_pct': failure_true,
                'difference': success_true - failure_true,
                'p_value': p_value,
                'significant': significant
            }
            
            # Create visualization
            if i < len(axes):
                data = [['Success', success_true], ['Success', 100-success_true], 
                        ['Failure', failure_true], ['Failure', 100-failure_true]]
                df = pd.DataFrame(data, columns=['Group', 'Percentage'])
                df['Type'] = ['True', 'False', 'True', 'False']
                
                sns.barplot(x='Group', y='Percentage', hue='Type', data=df, ax=axes[i])
                axes[i].set_title(f"{prop} Distribution\n" + 
                                (f"p={p_value:.4f} (significant)" if significant else 
                                 f"p={p_value:.4f} (not significant)" if p_value else "No statistical test"))
                axes[i].set_ylabel("Percentage (%)")
                
                # Add data labels
                axes[i].text(0-0.1, success_true/2, f'{success_true:.1f}%', ha='center')
                axes[i].text(0-0.1, success_true+((100-success_true)/2), f'{100-success_true:.1f}%', ha='center')
                axes[i].text(1+0.1, failure_true/2, f'{failure_true:.1f}%', ha='center')
                axes[i].text(1+0.1, failure_true+((100-failure_true)/2), f'{100-failure_true:.1f}%', ha='center')
                
                # Add sample size
                axes[i].text(0, -5, f'n={len(all_success_df)}', ha='center')
                axes[i].text(1, -5, f'n={len(all_failure_df)}', ha='center')
                
        # Hide empty subplots
        for i in range(len(available_binary), len(axes)):
            axes[i].axis('off')
            
        plt.suptitle(f"Binary Properties: Universal Success vs. Universal Failure", fontsize=16)
        plt.tight_layout()
        plt.show()
    
    # Create summary table of significant results
    significant_continuous = {k: v for k, v in continuous_results.items() if v.get('significant', False)}
    significant_discrete = {k: v for k, v in discrete_results.items() if v.get('significant', False)}
    significant_binary = {k: v for k, v in binary_results.items() if v.get('significant', False)}
    
    # Prepare unified significant results table for heatmap
    if significant_continuous or significant_discrete or significant_binary:
        plt.figure(figsize=(10, max(1, len(significant_continuous) + len(significant_binary))/2))
        
        # Prepare data for continuous and binary properties (discrete is more complex)
        labels = []
        values = []
        p_values = []
        
        for prop, stats in significant_continuous.items():
            labels.append(prop)
            values.append((stats['success_mean'] - stats['failure_mean']) / 
                         max(abs(stats['success_mean']), abs(stats['failure_mean'])) * 100)
            p_values.append(stats['p_value'])
            
        for prop, stats in significant_binary.items():
            labels.append(prop)
            values.append(stats['difference'])
            p_values.append(stats['p_value'])
            
        if labels:
            # Create a DataFrame for the plot
            df_plot = pd.DataFrame({
                'Property': labels, 
                'Difference (%)': values,
                'p-value': p_values
            })
            
            # Sort by absolute difference
            df_plot = df_plot.iloc[np.argsort(np.abs(df_plot['Difference (%)']))[::-1]]
            
            # With this code:
            ax = sns.barplot(y='Property', x='Difference (%)', data=df_plot)

            # Color the bars based on whether the difference is positive or negative
            for i, bar in enumerate(ax.patches):
                if df_plot['Difference (%)'].iloc[i] < 0:
                    bar.set_color('indianred')  # Red for negative
                else:
                    bar.set_color('steelblue')  # Blue for positive
                    
            # Set the x-axis limits to be symmetric around 0
            max_abs_diff = max(abs(df_plot['Difference (%)'].max()), abs(df_plot['Difference (%)'].min()))
            ax.set_xlim(-max_abs_diff*1.1, max_abs_diff*1.1)

            # Add a reference line at x=0
            ax.axvline(x=0, color='black', linestyle='-', linewidth=0.5)
            
            # Add p-value annotations
            for i, p_val in enumerate(df_plot['p-value']):
                plt.text(0, i, f'p={p_val:.4f}', ha='center', va='center', 
                        bbox=dict(facecolor='white', alpha=0.7))
                
            plt.title('Significant Property Differences Between Universal Success and Failure Cases')
            plt.tight_layout()
            plt.show()
    
    # Print statistical summaries
    if significant_continuous:
        print("\nSignificant differences in continuous properties:")
        for prop, stats in significant_continuous.items():
            print(f"{prop}: Success mean = {stats['success_mean']:.2f}, "
                  f"Failure mean = {stats['failure_mean']:.2f}, "
                  f"p-value = {stats['p_value']:.4f}")
    
    if significant_discrete:
        print("\nSignificant differences in discrete properties:")
        for prop, stats in significant_discrete.items():
            print(f"{prop}: p-value = {stats['p_value']:.4f}")
            print(f"  Success distribution: {dict(stats['success_percentage'].apply(lambda x: round(x, 1)))}")
            print(f"  Failure distribution: {dict(stats['failure_percentage'].apply(lambda x: round(x, 1)))}")
    
    if significant_binary:
        print("\nSignificant differences in binary properties:")
        for prop, stats in significant_binary.items():
            print(f"{prop}: Success = {stats['success_true_pct']:.1f}%, "
                  f"Failure = {stats['failure_true_pct']:.1f}%, "
                  f"difference = {stats['difference']:.1f}%, "
                  f"p-value = {stats['p_value']:.4f}")
    
    # Insights and conclusions
    print("\nKey insights:")
    insights = []
    
    for prop, stats in significant_continuous.items():
        if stats['success_mean'] > stats['failure_mean']:
            insights.append(f"- Higher {prop} is associated with universal success (Avg: {stats['success_mean']:.2f} vs {stats['failure_mean']:.2f})")
        else:
            insights.append(f"- Lower {prop} is associated with universal success (Avg: {stats['success_mean']:.2f} vs {stats['failure_mean']:.2f})")
            
    for prop, stats in significant_binary.items():
        if stats['success_true_pct'] > stats['failure_true_pct']:
            insights.append(f"- {'Presence of' if 'has_' in prop or 'is_' in prop else 'Having'} {prop} is associated with universal success" + 
                           f" ({stats['success_true_pct']:.1f}% vs {stats['failure_true_pct']:.1f}%)")
        else:
            insights.append(f"- {'Absence of' if 'has_' in prop or 'is_' in prop else 'Not having'} {prop} is associated with universal success" + 
                          f" ({100-stats['success_true_pct']:.1f}% vs {100-stats['failure_true_pct']:.1f}%)")
    
    if insights:
        for insight in insights:
            print(insight)
    else:
        print("No statistically significant property differences were found between universal success and failure cases.")
    
    # Return compiled results
    return {
        'all_success_proteins': all_success_proteins,
        'all_failure_proteins': all_failure_proteins,
        'success_matrix': success_matrix,
        'continuous_results': continuous_results,
        'discrete_results': discrete_results,
        'binary_results': binary_results,
        'significant_continuous': significant_continuous,
        'significant_discrete': significant_discrete,
        'significant_binary': significant_binary,
        'success_df': all_success_df,
        'failure_df': all_failure_df,
        'statistics': {
            'total_proteins': total_proteins,
            'all_success_count': all_success_count,
            'all_failure_count': all_failure_count,
            'all_success_percent': all_success_percent,
            'all_failure_percent': all_failure_percent
        }
    }

In [ ]:
methods_to_analyze = ['icm', 'diffdock_pocket_only', 'gnina', 'surfdock', 'chai-1']
results = analyze_universal_properties(
    df_combined, 
    methods=methods_to_analyze,
    rmsd_threshold=2.0, 
    annotation_df=test_annot_df, 
    inputs_df=inputs_csv
)


## Posebuster Analysis

In [ ]:
def analyze_posebusters_comparative(df_combined, method1, method2, rmsd_threshold=2.0):
    """
    Analyze PoseBusters metrics for cases where one method succeeds and the other fails,
    using McNemar's test to account for paired observations.
    
    Parameters:
    -----------
    df_combined : pandas.DataFrame
        Combined dataframe with results including PoseBusters metrics for all methods
    method1 : str
        Name of the first method to compare
    method2 : str
        Name of the second method to compare
    rmsd_threshold : float, optional
        RMSD threshold for considering a docking successful, default is 2.0 Å
        
    Returns:
    --------
    dict
        Dictionary containing analysis results and visualizations
    """
    import matplotlib.pyplot as plt
    import numpy as np
    import seaborn as sns
    import scipy.stats as stats
    from statsmodels.stats.contingency_tables import mcnemar
    import pandas as pd
    from matplotlib.colors import LinearSegmentedColormap
    
    # Define PoseBusters columns to analyze
    BUST_TEST_COLUMNS = [
        # chemical validity and consistency
        "mol_pred_loaded", "mol_true_loaded", "mol_cond_loaded", "sanitization",
        "molecular_formula", "molecular_bonds", "tetrahedral_chirality", 
        "double_bond_stereochemistry",
        # intramolecular validity
        "bond_lengths", "bond_angles", "internal_steric_clash",
        "aromatic_ring_flatness", "double_bond_flatness",
        # intermolecular validity
        "minimum_distance_to_protein", "minimum_distance_to_organic_cofactors",
        "minimum_distance_to_inorganic_cofactors", "volume_overlap_with_protein",
        "volume_overlap_with_organic_cofactors", "volume_overlap_with_inorganic_cofactors"
    ]
    
    # Create category mapping for visualization
    CATEGORY_MAPPING = {
        # chemical validity and consistency
        "mol_pred_loaded": "Chemical Validity",
        "mol_true_loaded": "Chemical Validity", 
        "mol_cond_loaded": "Chemical Validity",
        "sanitization": "Chemical Validity",
        "molecular_formula": "Chemical Validity",
        "molecular_bonds": "Chemical Validity",
        "tetrahedral_chirality": "Chemical Validity", 
        "double_bond_stereochemistry": "Chemical Validity",
        # intramolecular validity
        "bond_lengths": "Intramolecular Validity",
        "bond_angles": "Intramolecular Validity",
        "internal_steric_clash": "Intramolecular Validity",
        "aromatic_ring_flatness": "Intramolecular Validity",
        "double_bond_flatness": "Intramolecular Validity",
        # intermolecular validity
        "minimum_distance_to_protein": "Intermolecular Validity",
        "minimum_distance_to_organic_cofactors": "Intermolecular Validity",
        "minimum_distance_to_inorganic_cofactors": "Intermolecular Validity",
        "volume_overlap_with_protein": "Intermolecular Validity",
        "volume_overlap_with_organic_cofactors": "Intermolecular Validity",
        "volume_overlap_with_inorganic_cofactors": "Intermolecular Validity"
    }
    
    # First get the best RMSD per protein for each method
    df_method1 = df_combined[df_combined['method'] == method1]
    df_method2 = df_combined[df_combined['method'] == method2]

    best_rmsd_method1 = df_method1.groupby('protein').apply(
        lambda x: x.loc[x['rmsd'].idxmin()] if not x['rmsd'].isna().all() else x.iloc[0]
    ).reset_index(drop=True)
    
    best_rmsd_method2 = df_method2.groupby('protein').apply(
        lambda x: x.loc[x['rmsd'].idxmin()] if not x['rmsd'].isna().all() else x.iloc[0]
    ).reset_index(drop=True)
    
    # Create success column for each method
    best_rmsd_method1[f'rmsd_≤_{rmsd_threshold}å'] = best_rmsd_method1['rmsd'] <= rmsd_threshold
    best_rmsd_method2[f'rmsd_≤_{rmsd_threshold}å'] = best_rmsd_method2['rmsd'] <= rmsd_threshold
    
    # Find common proteins
    common_proteins = set(best_rmsd_method1['protein']).intersection(set(best_rmsd_method2['protein']))
    
    # Create a dataframe to track success/failure for each protein
    success_matrix = pd.DataFrame({'protein': list(common_proteins)})
    method1_success = {}
    method2_success = {}
    
    for protein in common_proteins:
        method1_success[protein] = best_rmsd_method1[
            best_rmsd_method1['protein'] == protein
        ][f'rmsd_≤_{rmsd_threshold}å'].iloc[0]
        
        method2_success[protein] = best_rmsd_method2[
            best_rmsd_method2['protein'] == protein
        ][f'rmsd_≤_{rmsd_threshold}å'].iloc[0]
    
    success_matrix[f'{method1}_success'] = success_matrix['protein'].map(method1_success)
    success_matrix[f'{method2}_success'] = success_matrix['protein'].map(method2_success)
    
    # Identify proteins where one method succeeds and the other fails
    method1_only_success = success_matrix[
        (success_matrix[f'{method1}_success']) & (~success_matrix[f'{method2}_success'])
    ]['protein'].tolist()
    
    method2_only_success = success_matrix[
        (~success_matrix[f'{method1}_success']) & (success_matrix[f'{method2}_success'])
    ]['protein'].tolist()
    
    # Extract PoseBusters results for the differential success cases
    method1_only_results = best_rmsd_method1[best_rmsd_method1['protein'].isin(method1_only_success)]
    method2_only_results = best_rmsd_method2[best_rmsd_method2['protein'].isin(method2_only_success)]
    
    # Check if PoseBusters columns are available in the data
    available_bust_columns = [col for col in BUST_TEST_COLUMNS if col in df_combined.columns]
    
    if len(available_bust_columns) == 0:
        print("No PoseBusters metrics found in the dataset. Cannot perform PoseBusters-specific analysis.")
        return None
    
    print(f"Analyzing PoseBusters metrics for {len(method1_only_success)} cases where {method1} succeeds but {method2} fails")
    print(f"and {len(method2_only_success)} cases where {method2} succeeds but {method1} fails")
    
    # Create a merged dataset for paired analysis
    merged_results = pd.merge(
        best_rmsd_method1[['protein'] + available_bust_columns], 
        best_rmsd_method2[['protein'] + available_bust_columns],
        on='protein',
        suffixes=(f'_{method1}', f'_{method2}')
    )
    
    # Analyze PoseBusters metrics
    posebusters_stats = {}
    
    for column in available_bust_columns:
        if column not in best_rmsd_method1.columns or column not in best_rmsd_method2.columns:
            continue
            
        # Calculate pass rates
        method1_pass_rate = method1_only_results[column].fillna(False).mean() * 100
        method2_pass_rate = method2_only_results[column].fillna(False).mean() * 100
        
        # Create the column names with method suffixes
        col1 = f"{column}_{method1}"
        col2 = f"{column}_{method2}"
        
        # Create a contingency table for McNemar's test
        # Format: [[both pass, method1 pass & method2 fail], [method1 fail & method2 pass, both fail]]
        contingency = np.zeros((2, 2), dtype=int)
        
        # Fill the contingency table
        both_pass = ((merged_results[col1].fillna(False)) & (merged_results[col2].fillna(False))).sum()
        m1_pass_m2_fail = ((merged_results[col1].fillna(False)) & (~merged_results[col2].fillna(False))).sum()
        m1_fail_m2_pass = ((~merged_results[col1].fillna(False)) & (merged_results[col2].fillna(False))).sum()
        both_fail = ((~merged_results[col1].fillna(False)) & (~merged_results[col2].fillna(False))).sum()
        
        contingency[0, 0] = both_pass
        contingency[0, 1] = m1_pass_m2_fail
        contingency[1, 0] = m1_fail_m2_pass
        contingency[1, 1] = both_fail
        
        # Calculate McNemar's test
        try:
            # Use exact=True for small sample sizes
            result = mcnemar(contingency, exact=True, correction=True)
            statistic = result.statistic
            p_value = result.pvalue
            significant = p_value < 0.05
        except:
            statistic, p_value, significant = None, None, False
        
        # Calculate odds ratio as effect size measure
        if m1_pass_m2_fail == 0 or m1_fail_m2_pass == 0:
            # Apply correction to avoid division by zero
            odds_ratio_value = ((m1_pass_m2_fail + 0.5) / (m1_fail_m2_pass + 0.5))
        else:
            odds_ratio_value = (m1_pass_m2_fail / m1_fail_m2_pass)
        
        posebusters_stats[column] = {
            f'{method1}_pass_rate': method1_pass_rate,
            f'{method2}_pass_rate': method2_pass_rate,
            'difference': method1_pass_rate - method2_pass_rate,
            'mcnemar_statistic': statistic,
            'p_value': p_value,
            'significant': significant,
            'category': CATEGORY_MAPPING.get(column, "Other"),
            'odds_ratio': odds_ratio_value,
            'contingency_table': contingency,
            'both_pass': both_pass,
            'm1_pass_m2_fail': m1_pass_m2_fail,
            'm1_fail_m2_pass': m1_fail_m2_pass,
            'both_fail': both_fail
        }
    
    # Organize metrics by category
    metrics_by_category = {}
    for metric, stats in posebusters_stats.items():
        category = stats['category']
        if category not in metrics_by_category:
            metrics_by_category[category] = []
        metrics_by_category[category].append(metric)
    
    # Create visualization by category
    if posebusters_stats:
        for category, metrics in metrics_by_category.items():
            if not metrics:
                continue
                
            plt.figure(figsize=(12, max(6, len(metrics) * 0.5)))
            metrics.sort(key=lambda x: posebusters_stats[x]['difference'], reverse=True)
            
            method1_rates = [posebusters_stats[m][f'{method1}_pass_rate'] for m in metrics]
            method2_rates = [posebusters_stats[m][f'{method2}_pass_rate'] for m in metrics]
            
            x = np.arange(len(metrics))
            width = 0.35
            
            bars1 = plt.bar(x - width/2, method1_rates, width, label=f'{method1} only success', color='blue', alpha=0.7)
            bars2 = plt.bar(x + width/2, method2_rates, width, label=f'{method2} only success', color='orange', alpha=0.7)
            
            # Add significance markers
            for i, metric in enumerate(metrics):
                if posebusters_stats[metric]['significant']:
                    plt.text(i, max(method1_rates[i], method2_rates[i]) + 3, '*', 
                             ha='center', fontsize=14, fontweight='bold')
            
            plt.xlabel('PoseBusters Metric')
            plt.ylabel('Pass Rate (%)')
            plt.title(f'{category} Metrics: {method1} vs {method2} Unique Successes')
            plt.xticks(x, [m.replace('_', ' ').title() for m in metrics], rotation=45, ha='right')
            plt.legend()
            plt.grid(axis='y', alpha=0.3)
            
            # Add data labels
            for i, v in enumerate(method1_rates):
                plt.text(i - width/2, v + 1, f'{v:.1f}%', ha='center')
            for i, v in enumerate(method2_rates):
                plt.text(i + width/2, v + 1, f'{v:.1f}%', ha='center')
                
            plt.tight_layout()
            plt.show()
        
        # Create heatmap for difference in pass rates
        plt.figure(figsize=(12, 8))
        
        # Organize data for heatmap
        categories = list(metrics_by_category.keys())
        max_metrics = max(len(metrics) for metrics in metrics_by_category.values())
        
        # Create data matrix for heatmap (difference in pass rates)
        heatmap_data = np.zeros((len(categories), max_metrics))
        heatmap_data.fill(np.nan)  # Fill with NaN to hide empty cells
        
        # X and Y tick labels
        y_labels = []
        x_labels = []
        
        for i, category in enumerate(categories):
            metrics = metrics_by_category[category]
            metrics.sort(key=lambda x: abs(posebusters_stats[x]['difference']), reverse=True)
            
            for j, metric in enumerate(metrics):
                heatmap_data[i, j] = posebusters_stats[metric]['difference']
                
                # Add metric label if this is the first category
                if i == 0:
                    x_labels.append(metric.replace('_', ' ').title())
                    
            # Add category label
            y_labels.append(category)
            
            # Fill unused cells with NaN
            for j in range(len(metrics), max_metrics):
                heatmap_data[i, j] = np.nan
        
        # Define custom colormap (blue for positive difference, orange for negative)
        colors = ["orange", "white", "blue"]
        n_bins = 100
        cmap = LinearSegmentedColormap.from_list("custom_diverging", colors, N=n_bins)
        
        # Create heatmap
        ax = sns.heatmap(
            heatmap_data, 
            cmap=cmap,
            center=0,
            annot=True, 
            fmt=".1f",
            linewidths=.5, 
            yticklabels=y_labels,
            mask=np.isnan(heatmap_data),  # Mask NaN values
            cbar_kws={'label': f'Difference in Pass Rate (%) ({method1} - {method2})'}
        )
        
        # Set x-tick labels
        x_ticks = np.arange(0.5, max_metrics)
        ax.set_xticks(x_ticks)
        ax.set_xticklabels(x_labels, rotation=45, ha='right')
        
        plt.title(f'Difference in PoseBusters Pass Rates: {method1} vs {method2}')
        plt.tight_layout()
        plt.show()
    
    # Summary of significant differences
    significant_metrics = {k: v for k, v in posebusters_stats.items() if v.get('significant', False)}
    
    if significant_metrics:
        print("\nSignificant differences in PoseBusters metrics (McNemar's test):")
        for metric, stats in significant_metrics.items():
            print(f"{metric}: {method1} = {stats[f'{method1}_pass_rate']:.1f}%, "
                  f"{method2} = {stats[f'{method2}_pass_rate']:.1f}%, "
                  f"difference = {stats['difference']:.1f}%, "
                  f"p-value = {stats['p_value']:.4f}, "
                  f"odds ratio = {stats['odds_ratio']:.2f}")
            print(f"  Contingency: Both pass: {stats['both_pass']}, "
                  f"{method1} only: {stats['m1_pass_m2_fail']}, "
                  f"{method2} only: {stats['m1_fail_m2_pass']}, "
                  f"Both fail: {stats['both_fail']}")
    else:
        print("\nNo statistically significant differences found in PoseBusters metrics.")
    
    # Insights and conclusions
    if significant_metrics:
        print("\nInsights:")
        
        # Group insights by category
        insights_by_category = {}
        for metric, stats in significant_metrics.items():
            category = stats['category']
            if category not in insights_by_category:
                insights_by_category[category] = []
            
            if stats['difference'] > 0:
                insights_by_category[category].append(
                    f"{method1} performs better on {metric} validation ({stats[f'{method1}_pass_rate']:.1f}% vs {stats[f'{method2}_pass_rate']:.1f}%)"
                )
            else:
                insights_by_category[category].append(
                    f"{method2} performs better on {metric} validation ({stats[f'{method2}_pass_rate']:.1f}% vs {stats[f'{method1}_pass_rate']:.1f}%)"
                )
        
        for category, insights in insights_by_category.items():
            print(f"\n{category}:")
            for insight in insights:
                print(f"- {insight}")
    
    return {
        'posebusters_stats': posebusters_stats,
        'method1_only_proteins': method1_only_success,
        'method2_only_proteins': method2_only_success,
        'method1_only_results': method1_only_results,
        'method2_only_results': method2_only_results,
        'significant_metrics': significant_metrics,
        'categories': metrics_by_category
    }

In [ ]:
def analyze_posebusters_universal(df_combined, methods, rmsd_threshold=2.0):
    """
    Analyze PoseBusters metrics for cases where all methods succeed or all methods fail.
    
    Parameters:
    -----------
    df_combined : pandas.DataFrame
        Combined dataframe with results including PoseBusters metrics for all methods
    methods : list
        List of method names to analyze
    rmsd_threshold : float, optional
        RMSD threshold for considering a docking successful, default is 2.0 Å
        
    Returns:
    --------
    dict
        Dictionary containing analysis results and visualizations
    """
    import matplotlib.pyplot as plt
    import numpy as np
    import seaborn as sns
    import scipy.stats as stats
    import pandas as pd
    
    # Define PoseBusters columns to analyze
    BUST_TEST_COLUMNS = [
        # chemical validity and consistency
        "mol_pred_loaded", "mol_true_loaded", "mol_cond_loaded", "sanitization",
        "molecular_formula", "molecular_bonds", "tetrahedral_chirality", 
        "double_bond_stereochemistry",
        # intramolecular validity
        "bond_lengths", "bond_angles", "internal_steric_clash",
        "aromatic_ring_flatness", "double_bond_flatness",
        # intermolecular validity
        "minimum_distance_to_protein", "minimum_distance_to_organic_cofactors",
        "minimum_distance_to_inorganic_cofactors", "volume_overlap_with_protein",
        "volume_overlap_with_organic_cofactors", "volume_overlap_with_inorganic_cofactors"
    ]
    
    # Create category mapping for visualization
    CATEGORY_MAPPING = {
        # chemical validity and consistency
        "mol_pred_loaded": "Chemical Validity",
        "mol_true_loaded": "Chemical Validity", 
        "mol_cond_loaded": "Chemical Validity",
        "sanitization": "Chemical Validity",
        "molecular_formula": "Chemical Validity",
        "molecular_bonds": "Chemical Validity",
        "tetrahedral_chirality": "Chemical Validity", 
        "double_bond_stereochemistry": "Chemical Validity",
        # intramolecular validity
        "bond_lengths": "Intramolecular Validity",
        "bond_angles": "Intramolecular Validity",
        "internal_steric_clash": "Intramolecular Validity",
        "aromatic_ring_flatness": "Intramolecular Validity",
        "double_bond_flatness": "Intramolecular Validity",
        # intermolecular validity
        "minimum_distance_to_protein": "Intermolecular Validity",
        "minimum_distance_to_organic_cofactors": "Intermolecular Validity",
        "minimum_distance_to_inorganic_cofactors": "Intermolecular Validity",
        "volume_overlap_with_protein": "Intermolecular Validity",
        "volume_overlap_with_organic_cofactors": "Intermolecular Validity",
        "volume_overlap_with_inorganic_cofactors": "Intermolecular Validity"
    }
    
    # Get best RMSD per protein for each method
    best_rmsd_by_method = {}
    for method in methods:
        df_method = df_combined[df_combined['method'] == method]
        best_rmsd = df_method.groupby('protein').apply(
            lambda x: x.loc[x['rmsd'].idxmin()] if not x['rmsd'].isna().all() else x.iloc[0]
        ).reset_index(drop=True)
        
        # Create success column
        best_rmsd[f'rmsd_≤_{rmsd_threshold}å'] = best_rmsd['rmsd'] <= rmsd_threshold
        
        best_rmsd_by_method[method] = best_rmsd
    
    # Find common proteins across all methods
    common_proteins = set(best_rmsd_by_method[methods[0]]['protein'])
    for method in methods[1:]:
        common_proteins = common_proteins.intersection(set(best_rmsd_by_method[method]['protein']))
    common_proteins = list(common_proteins)
    
    # Create a dataframe to track success/failure for each protein across all methods
    success_matrix = pd.DataFrame({'protein': common_proteins})
    for method in methods:
        method_success = {}
        for protein in common_proteins:
            success = best_rmsd_by_method[method][
                best_rmsd_by_method[method]['protein'] == protein
            ][f'rmsd_≤_{rmsd_threshold}å'].iloc[0]
            method_success[protein] = success
        success_matrix[f'{method}_success'] = success_matrix['protein'].map(method_success)
    
    # Calculate total successes for each protein
    success_matrix['total_successes'] = success_matrix[[f'{method}_success' for method in methods]].sum(axis=1)
    
    # Identify proteins where all methods succeed or all methods fail
    all_success_proteins = success_matrix[success_matrix['total_successes'] == len(methods)]['protein'].tolist()
    all_failure_proteins = success_matrix[success_matrix['total_successes'] == 0]['protein'].tolist()
    
    print(f"Analyzing PoseBusters metrics for {len(all_success_proteins)} cases where all methods succeed")
    print(f"and {len(all_failure_proteins)} cases where all methods fail")
    
    # Check if we have enough data to analyze
    if len(all_success_proteins) < 5 or len(all_failure_proteins) < 5:
        print("Warning: Small sample size may affect statistical significance.")
    
    # Get PoseBusters results for these proteins
    all_success_results = pd.DataFrame()
    all_failure_results = pd.DataFrame()
    
    # For each method, collect the PoseBusters data
    for method in methods:
        method_success = best_rmsd_by_method[method][best_rmsd_by_method[method]['protein'].isin(all_success_proteins)]
        method_failure = best_rmsd_by_method[method][best_rmsd_by_method[method]['protein'].isin(all_failure_proteins)]
        
        all_success_results = pd.concat([all_success_results, method_success])
        all_failure_results = pd.concat([all_failure_results, method_failure])
    
    # Check if PoseBusters columns are available in the data
    available_bust_columns = [col for col in BUST_TEST_COLUMNS if col in df_combined.columns]
    
    if len(available_bust_columns) == 0:
        print("No PoseBusters metrics found in the dataset. Cannot perform PoseBusters-specific analysis.")
        return None
    
    # Analyze PoseBusters metrics
    posebusters_stats = {}
    
    for column in available_bust_columns:
        if column not in all_success_results.columns or column not in all_failure_results.columns:
            continue
            
        # Calculate pass rates
        success_pass_rate = all_success_results[column].mean() * 100
        failure_pass_rate = all_failure_results[column].mean() * 100
        
        # Fisher's exact test
        contingency = [
            [all_success_results[column].sum(), len(all_success_results) - all_success_results[column].sum()],
            [all_failure_results[column].sum(), len(all_failure_results) - all_failure_results[column].sum()]
        ]
        
        try:
            _, p_value = stats.fisher_exact(contingency)
            significant = p_value < 0.05
        except:
            p_value, significant = None, False
        
        posebusters_stats[column] = {
            'success_pass_rate': success_pass_rate,
            'failure_pass_rate': failure_pass_rate,
            'difference': success_pass_rate - failure_pass_rate,
            'p_value': p_value,
            'significant': significant,
            'category': CATEGORY_MAPPING.get(column, "Other")
        }
    
    # Organize metrics by category
    metrics_by_category = {}
    for metric, stats in posebusters_stats.items():
        category = stats['category']
        if category not in metrics_by_category:
            metrics_by_category[category] = []
        metrics_by_category[category].append(metric)
    
    # Create visualization by category
    if posebusters_stats:
        for category, metrics in metrics_by_category.items():
            if not metrics:
                continue
                
            plt.figure(figsize=(12, max(6, len(metrics) * 0.5)))
            metrics.sort(key=lambda x: posebusters_stats[x]['difference'], reverse=True)
            
            success_rates = [posebusters_stats[m]['success_pass_rate'] for m in metrics]
            failure_rates = [posebusters_stats[m]['failure_pass_rate'] for m in metrics]
            
            x = np.arange(len(metrics))
            width = 0.35
            
            bars1 = plt.bar(x - width/2, success_rates, width, label='All methods succeed', color='green', alpha=0.7)
            bars2 = plt.bar(x + width/2, failure_rates, width, label='All methods fail', color='red', alpha=0.7)
            
            # Add significance markers
            for i, metric in enumerate(metrics):
                if posebusters_stats[metric]['significant']:
                    plt.text(i, max(success_rates[i], failure_rates[i]) + 3, '*', 
                             ha='center', fontsize=14, fontweight='bold')
            
            plt.xlabel('PoseBusters Metric')
            plt.ylabel('Pass Rate (%)')
            plt.title(f'{category} Metrics: Universal Success vs. Universal Failure')
            plt.xticks(x, [m.replace('_', ' ').title() for m in metrics], rotation=45, ha='right')
            plt.legend()
            plt.grid(axis='y', alpha=0.3)
            
            # Add data labels
            for i, v in enumerate(success_rates):
                plt.text(i - width/2, v + 1, f'{v:.1f}%', ha='center')
            for i, v in enumerate(failure_rates):
                plt.text(i + width/2, v + 1, f'{v:.1f}%', ha='center')
                
            plt.tight_layout()
            plt.show()
    
    # Create summary heatmap
    plt.figure(figsize=(14, 10))
    
    # Prepare data for heatmap
    metric_labels = []
    difference_values = []
    significance = []
    categories = []
    
    for metric, stats in posebusters_stats.items():
        metric_labels.append(metric.replace('_', ' ').title())
        difference_values.append(stats['difference'])
        significance.append(stats['significant'])
        categories.append(stats['category'])
    
    # Create a dataframe for the heatmap
    heatmap_df = pd.DataFrame({
        'Metric': metric_labels,
        'Difference': difference_values,
        'Significant': significance,
        'Category': categories
    })
    
    # Sort by category and then by difference
    heatmap_df = heatmap_df.sort_values(['Category', 'Difference'], ascending=[True, False])
    
    # Create the heatmap
    plt.figure(figsize=(10, max(8, len(heatmap_df) * 0.4)))
    heatmap = sns.heatmap(
        heatmap_df.pivot_table(
            index='Metric',
            values='Difference',
            aggfunc='first'
        ).sort_index(),
        cmap='RdBu_r',
        center=0,
        annot=True,
        fmt='.1f',
        cbar_kws={'label': 'Difference in Pass Rate (%) (Success - Failure)'}
    )
    
    # Add asterisks for significant differences
    for i, sig in enumerate(heatmap_df['Significant']):
        if sig:
            plt.text(1.05, i + 0.5, '*', fontsize=14, fontweight='bold')
    
    plt.title('Difference in PoseBusters Pass Rates: Universal Success vs. Universal Failure')
    plt.tight_layout()
    plt.show()
    
    # Summary of significant differences
    significant_metrics = {k: v for k, v in posebusters_stats.items() if v.get('significant', False)}
    
    if significant_metrics:
        print("\nSignificant differences in PoseBusters metrics:")
        for metric, stats in significant_metrics.items():
            print(f"{metric}: Success = {stats['success_pass_rate']:.1f}%, "
                  f"Failure = {stats['failure_pass_rate']:.1f}%, "
                  f"difference = {stats['difference']:.1f}%, "
                  f"p-value = {stats['p_value']:.4f}")
    else:
        print("\nNo statistically significant differences found in PoseBusters metrics.")
    
    # Insights and conclusions
    if significant_metrics:
        print("\nInsights by category:")
        
        # Group insights by category
        insights_by_category = {}
        for metric, stats in significant_metrics.items():
            category = stats['category']
            if category not in insights_by_category:
                insights_by_category[category] = []
            
            if stats['difference'] > 0:
                insights_by_category[category].append(
                    f"Higher pass rate for {metric} is associated with universal success ({stats['success_pass_rate']:.1f}% vs {stats['failure_pass_rate']:.1f}%)"
                )
            else:
                insights_by_category[category].append(
                    f"Lower pass rate for {metric} is associated with universal success ({stats['success_pass_rate']:.1f}% vs {stats['failure_pass_rate']:.1f}%)"
                )
        
        for category, insights in insights_by_category.items():
            print(f"\n{category}:")
            for insight in insights:
                print(f"- {insight}")
        
        # Overall patterns
        chemical_validity_metrics = [m for m in significant_metrics if posebusters_stats[m]['category'] == 'Chemical Validity']
        intramolecular_metrics = [m for m in significant_metrics if posebusters_stats[m]['category'] == 'Intramolecular Validity']
        intermolecular_metrics = [m for m in significant_metrics if posebusters_stats[m]['category'] == 'Intermolecular Validity']
        
        print("\nOverall patterns:")
        if chemical_validity_metrics:
            chemical_better = sum(1 for m in chemical_validity_metrics if posebusters_stats[m]['difference'] > 0)
            if chemical_better > len(chemical_validity_metrics) / 2:
                print("- Universal success cases generally have better chemical validity")
            elif chemical_better < len(chemical_validity_metrics) / 2:
                print("- Universal failure cases surprisingly show better chemical validity in some metrics")
        
        if intramolecular_metrics:
            intra_better = sum(1 for m in intramolecular_metrics if posebusters_stats[m]['difference'] > 0)
            if intra_better > len(intramolecular_metrics) / 2:
                print("- Universal success cases generally have better intramolecular validity")
            elif intra_better < len(intramolecular_metrics) / 2:
                print("- Universal failure cases surprisingly show better intramolecular validity in some metrics")
        
        if intermolecular_metrics:
            inter_better = sum(1 for m in intermolecular_metrics if posebusters_stats[m]['difference'] > 0)
            if inter_better > len(intermolecular_metrics) / 2:
                print("- Universal success cases generally have better intermolecular validity")
            elif inter_better < len(intermolecular_metrics) / 2:
                print("- Universal failure cases surprisingly show better intermolecular validity in some metrics")
    
    return {
        'posebusters_stats': posebusters_stats,
        'all_success_proteins': all_success_proteins,
        'all_failure_proteins': all_failure_proteins,
        'all_success_results': all_success_results,
        'all_failure_results': all_failure_results,
        'significant_metrics': significant_metrics,
        'categories': metrics_by_category
    }

In [ ]:
# Analyze PoseBusters metrics for cases where one method succeeds and the other fails
posebusters_comparative_analysis = analyze_posebusters_comparative(
    df_combined, 
    method1='icm', 
    method2='diffdock_pocket_only', 
    rmsd_threshold=2.0
)

# Extract key insights
significant_metrics = posebusters_comparative_analysis['significant_metrics']
print(f"Found {len(significant_metrics)} significantly different PoseBusters metrics")

# Get list of proteins where method1 succeeds but method2 fails for further inspection
method1_only_proteins = posebusters_comparative_analysis['method1_only_proteins']

In [ ]:
# Analyze PoseBusters metrics for cases where one method succeeds and the other fails
posebusters_comparative_analysis = analyze_posebusters_comparative(
    df_combined, 
    method1='icm', 
    method2='surfdock', 
    rmsd_threshold=2.0
)

# Extract key insights
significant_metrics = posebusters_comparative_analysis['significant_metrics']
print(f"Found {len(significant_metrics)} significantly different PoseBusters metrics")

# Get list of proteins where method1 succeeds but method2 fails for further inspection
method1_only_proteins = posebusters_comparative_analysis['method1_only_proteins']

In [ ]:
# Analyze PoseBusters metrics for cases where one method succeeds and the other fails
posebusters_comparative_analysis = analyze_posebusters_comparative(
    df_combined, 
    method1='icm', 
    method2='chai-1', 
    rmsd_threshold=2.0
)

# Extract key insights
significant_metrics = posebusters_comparative_analysis['significant_metrics']
print(f"Found {len(significant_metrics)} significantly different PoseBusters metrics")

# Get list of proteins where method1 succeeds but method2 fails for further inspection
method1_only_proteins = posebusters_comparative_analysis['method1_only_proteins']

In [ ]:
analyze_posebusters_comparative(
    df_combined, 
    method1='surfdock', 
    method2='chai-1', 
    rmsd_threshold=2.0
)

In [ ]:
# Analyze PoseBusters metrics for cases where all methods succeed or fail
methods_to_analyze = ['icm', 'diffdock', 'gnina', 'surfdock']

posebusters_universal_analysis = analyze_posebusters_universal(
    df_combined, 
    methods=methods_to_analyze,
    rmsd_threshold=2.0
)

# Extract key insights
significant_metrics = posebusters_universal_analysis['significant_metrics']
print(f"Found {len(significant_metrics)} significantly different PoseBusters metrics")

# Get proteins that are universally "easy" or "difficult" for further inspection
universally_easy = posebusters_universal_analysis['all_success_proteins']
universally_difficult = posebusters_universal_analysis['all_failure_proteins']

### posebuester analysis for individual methods

In [ ]:
def analyze_posebusters_by_method(df_combined, method, rmsd_threshold=2.0, imbalance_threshold=0.05):
    """
    Analyze how PoseBusters validity metrics affect success rate of a single docking method,
    with special handling for imbalanced metrics.
    
    Parameters:
    -----------
    df_combined : pandas.DataFrame
        Combined dataframe with results including PoseBusters metrics
    method : str
        Name of the docking method to analyze
    rmsd_threshold : float, optional
        RMSD threshold for considering a docking successful, default is 2.0 Å
    imbalance_threshold : float, optional
        Threshold ratio below which a metric is considered severely imbalanced (e.g., 0.05 means 
        if either true/false class represents <5% of data, it's considered imbalanced)
        
    Returns:
    --------
    dict
        Dictionary containing analysis results by validity category
    """
    import matplotlib.pyplot as plt
    import numpy as np
    import seaborn as sns
    import scipy.stats as stats
    import pandas as pd
    from matplotlib.colors import LinearSegmentedColormap
    
    # Filter data for the specified method
    df_method = df_combined[df_combined['method'] == method]
    
    # Get best RMSD per protein for the method
    best_rmsd_df = df_method.groupby('protein').apply(
        lambda x: x.loc[x['rmsd'].idxmin()] if not x['rmsd'].isna().all() else x.iloc[0]
    ).reset_index(drop=True)
    
    # Add success column if not present
    if f'rmsd_≤_{rmsd_threshold}å' not in best_rmsd_df.columns:
        best_rmsd_df[f'rmsd_≤_{rmsd_threshold}å'] = best_rmsd_df['rmsd'] <= rmsd_threshold
    
    # Separate successful and unsuccessful cases
    successful = best_rmsd_df[best_rmsd_df[f'rmsd_≤_{rmsd_threshold}å']]
    unsuccessful = best_rmsd_df[~best_rmsd_df[f'rmsd_≤_{rmsd_threshold}å']]
    
    success_rate = len(successful) / len(best_rmsd_df) * 100
    print(f"Method: {method}")
    print(f"Total proteins: {len(best_rmsd_df)}")
    print(f"Success rate (RMSD ≤ {rmsd_threshold}Å): {success_rate:.2f}%")
    
    # Define PoseBusters columns to analyze
    BUST_TEST_COLUMNS = [
        # chemical validity and consistency
        "mol_pred_loaded", "mol_true_loaded", "mol_cond_loaded", "sanitization",
        "molecular_formula", "molecular_bonds", "tetrahedral_chirality", 
        "double_bond_stereochemistry",
        # intramolecular validity
        "bond_lengths", "bond_angles", "internal_steric_clash",
        "aromatic_ring_flatness", "double_bond_flatness",
        # intermolecular validity
        "minimum_distance_to_protein", "minimum_distance_to_organic_cofactors",
        "minimum_distance_to_inorganic_cofactors", "volume_overlap_with_protein",
        "volume_overlap_with_organic_cofactors", "volume_overlap_with_inorganic_cofactors"
    ]
    
    # Create category mapping
    CATEGORY_MAPPING = {
        # chemical validity and consistency
        "mol_pred_loaded": "Chemical Validity",
        "mol_true_loaded": "Chemical Validity", 
        "mol_cond_loaded": "Chemical Validity",
        "sanitization": "Chemical Validity",
        "molecular_formula": "Chemical Validity",
        "molecular_bonds": "Chemical Validity",
        "tetrahedral_chirality": "Chemical Validity", 
        "double_bond_stereochemistry": "Chemical Validity",
        # intramolecular validity
        "bond_lengths": "Intramolecular Validity",
        "bond_angles": "Intramolecular Validity",
        "internal_steric_clash": "Intramolecular Validity",
        "aromatic_ring_flatness": "Intramolecular Validity",
        "double_bond_flatness": "Intramolecular Validity",
        # intermolecular validity
        "minimum_distance_to_protein": "Intermolecular Validity",
        "minimum_distance_to_organic_cofactors": "Intermolecular Validity",
        "minimum_distance_to_inorganic_cofactors": "Intermolecular Validity",
        "volume_overlap_with_protein": "Intermolecular Validity",
        "volume_overlap_with_organic_cofactors": "Intermolecular Validity",
        "volume_overlap_with_inorganic_cofactors": "Intermolecular Validity"
    }
    
    # Check available columns
    available_bust_columns = [col for col in BUST_TEST_COLUMNS if col in best_rmsd_df.columns]
    
    if len(available_bust_columns) == 0:
        print("No PoseBusters metrics found in the dataset.")
        return None
    
    # Analyze metrics
    posebusters_stats = {}
    
    for column in available_bust_columns:
        # Calculate class balance
        column_values = best_rmsd_df[column].fillna(False)
        true_count = column_values.sum()
        false_count = len(column_values) - true_count
        true_ratio = true_count / len(column_values)
        false_ratio = false_count / len(column_values)
        
        # Check for severe imbalance
        is_imbalanced = min(true_ratio, false_ratio) < imbalance_threshold
        
        # Calculate pass rates
        success_pass_rate = successful[column].fillna(False).mean() * 100 if len(successful) > 0 else 0
        fail_pass_rate = unsuccessful[column].fillna(False).mean() * 100 if len(unsuccessful) > 0 else 0
        
        # Calculate success rates for poses that pass and fail this metric
        pass_this_metric = best_rmsd_df[best_rmsd_df[column].fillna(False)]
        fail_this_metric = best_rmsd_df[~best_rmsd_df[column].fillna(False)]
        
        # Get counts for each category
        pass_count = len(pass_this_metric)
        fail_count = len(fail_this_metric)
        
        # Calculate success rates with handling for empty groups
        success_rate_when_pass = pass_this_metric[f'rmsd_≤_{rmsd_threshold}å'].mean() * 100 if pass_count > 0 else 0
        success_rate_when_fail = fail_this_metric[f'rmsd_≤_{rmsd_threshold}å'].mean() * 100 if fail_count > 0 else 0
        success_rate_difference = success_rate_when_pass - success_rate_when_fail
        
        # Fisher's exact test
        contingency = [
            [pass_this_metric[f'rmsd_≤_{rmsd_threshold}å'].sum(), 
             pass_count - pass_this_metric[f'rmsd_≤_{rmsd_threshold}å'].sum()],
            [fail_this_metric[f'rmsd_≤_{rmsd_threshold}å'].sum(), 
             fail_count - fail_this_metric[f'rmsd_≤_{rmsd_threshold}å'].sum()]
        ]
        
        try:
            _, p_value = stats.fisher_exact(contingency)
            significant = p_value < 0.05
            
            # If imbalanced, adjust significance interpretation
            if is_imbalanced:
                # Don't automatically discard significance, but flag it
                significance_reliable = False
            else:
                significance_reliable = True
        except:
            p_value, significant, significance_reliable = None, False, False
        
        posebusters_stats[column] = {
            'success_pass_rate': success_pass_rate,
            'fail_pass_rate': fail_pass_rate,
            'metric_difference': success_pass_rate - fail_pass_rate,
            'success_rate_when_pass': success_rate_when_pass,
            'success_rate_when_fail': success_rate_when_fail,
            'success_rate_difference': success_rate_difference,
            'p_value': p_value,
            'significant': significant,
            'significance_reliable': significance_reliable,
            'category': CATEGORY_MAPPING.get(column, "Other"),
            'pass_count': pass_count,
            'fail_count': fail_count,
            'is_imbalanced': is_imbalanced,
            'balance_ratio': min(true_ratio, false_ratio) / max(true_ratio, false_ratio)
        }
    
    # Organize by category
    metrics_by_category = {}
    for metric, stats in posebusters_stats.items():
        category = stats['category']
        if category not in metrics_by_category:
            metrics_by_category[category] = []
        metrics_by_category[category].append(metric)
    
    # Visualize results by category
    for category, metrics in metrics_by_category.items():
        if not metrics:
            continue
        
        # Sort metrics by success rate difference
        metrics.sort(key=lambda x: posebusters_stats[x]['success_rate_difference'], reverse=True)
        
        # Create figure
        plt.figure(figsize=(12, max(6, len(metrics) * 0.5)))
        
        # Prepare data for the plot
        labels = []
        differences = []
        significant = []
        is_imbalanced = []
        
        for m in metrics:
            stat = posebusters_stats[m]
            # Add sample size to label for context
            labels.append(f"{m.replace('_', ' ').title()} (P:{stat['pass_count']}/F:{stat['fail_count']})")
            differences.append(stat['success_rate_difference'])
            significant.append(stat['significant'])
            is_imbalanced.append(stat['is_imbalanced'])
        
        # Create bar chart with lighter color for imbalanced metrics
        colors = []
        for i, diff in enumerate(differences):
            if is_imbalanced[i]:
                # Use lighter colors for imbalanced metrics
                colors.append('lightgreen' if diff > 0 else 'lightcoral')
            else:
                colors.append('green' if diff > 0 else 'red')
        
        bars = plt.barh(labels, differences, color=colors)
        
        # Add significance markers with different symbol for imbalanced metrics
        for i, (sig, imb) in enumerate(zip(significant, is_imbalanced)):
            if sig:
                marker = '?' if imb else '*'
                plt.text(0, i, marker, ha='center', fontsize=14, fontweight='bold')
        
        # Add reference line and labels
        plt.axvline(x=0, color='black', linestyle='-', linewidth=0.5)
        plt.xlabel('Difference in Success Rate (% points) when metric passes vs. fails')
        plt.title(f'{category} Metrics Impact on Success for {method}')
        plt.grid(axis='x', alpha=0.3)
        
        # Add legend for imbalanced metrics
        legend_elements = [
            plt.Rectangle((0,0),1,1, color='green', label='Balanced'),
            plt.Rectangle((0,0),1,1, color='lightgreen', label='Imbalanced (<5% in one class)'),
            plt.Line2D([0], [0], marker='*', color='w', markerfacecolor='black', 
                      markersize=10, label='Significant'),
            plt.Line2D([0], [0], marker='D', color='w', markerfacecolor='black', 
                      markersize=10, label='Significant but unreliable')
        ]
        plt.legend(handles=legend_elements, loc='lower right')
        
        # Add data labels with confidence indicator
        for i, bar in enumerate(bars):
            value = differences[i]
            confidence = '' if not is_imbalanced[i] else ' (low confidence)'
            plt.text(value + (2 if value >= 0 else -2), 
                    i, 
                    f'{value:.1f}%{confidence}', 
                    ha='left' if value >= 0 else 'right',
                    va='center',
                    color='black')
        
        plt.tight_layout()
        plt.show()
    
    # Summary of significant metrics
    significant_metrics = {k: v for k, v in posebusters_stats.items() if v.get('significant', False)}
    
    print("\nMetrics with significant impact on success rate:")
    for category in ["Chemical Validity", "Intramolecular Validity", "Intermolecular Validity"]:
        category_metrics = [m for m, stats in significant_metrics.items() 
                          if stats['category'] == category]
        
        if category_metrics:
            print(f"\n{category}:")
            for metric in category_metrics:
                stats = posebusters_stats[metric]
                confidence_note = " (LOW CONFIDENCE due to imbalance)" if stats['is_imbalanced'] else ""
                print(f"- {metric}: Success rate {stats['success_rate_when_pass']:.1f}% when pass vs. "
                      f"{stats['success_rate_when_fail']:.1f}% when fail (diff: {stats['success_rate_difference']:.1f}%, "
                      f"p={stats['p_value']:.4f}, pass/fail counts: {stats['pass_count']}/{stats['fail_count']}){confidence_note}")
        else:
            print(f"\n{category}: No significant metrics found")
    
    # Print summary of imbalanced metrics
    imbalanced_metrics = {k: v for k, v in posebusters_stats.items() if v.get('is_imbalanced', False)}
    if imbalanced_metrics:
        print("\nNote: The following metrics have severe class imbalance (one class < 5% of data):")
        for metric, stats in imbalanced_metrics.items():
            min_class = "pass" if stats['pass_count'] < stats['fail_count'] else "fail"
            min_count = min(stats['pass_count'], stats['fail_count'])
            total = stats['pass_count'] + stats['fail_count']
            percent = (min_count / total) * 100
            print(f"- {metric}: Only {min_count} samples ({percent:.2f}%) {min_class} this metric")
    
    # Modified results with imbalance information
    return {
        'method': method,
        'overall_success_rate': success_rate,
        'metrics_stats': posebusters_stats,
        'metrics_by_category': metrics_by_category,
        'significant_metrics': significant_metrics,
        'imbalanced_metrics': imbalanced_metrics
    }

In [ ]:
df_method = df_combined[df_combined['method'] == 'icm']
best_rmsd_df = df_method.groupby('protein').apply(
    lambda x: x.loc[x['rmsd'].idxmin()] if not x['rmsd'].isna().all() else x.iloc[0]
).reset_index(drop=True)

In [ ]:
best_rmsd_df['internal_steric_clash'].value_counts()

In [ ]:
# List of methods to analyze
methods = ['icm', 'diffdock_pocket_only', 'gnina', 'surfdock', 'chai-1']

# Analyze each method
results = {}
for method in methods:
    print(f"\n{'='*50}\nAnalyzing {method}\n{'='*50}")
    results[method] = analyze_posebusters_by_method(df_combined, method, rmsd_threshold=2.0)